In [227]:
import sys
from pathlib import Path

# climb up until we find the folder that contains "src"
p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent

sys.path.insert(0, str(p))  # add project root to Python path
print("Project root:", p)


Project root: /Users/wenxi/Desktop/TFM_25


In [228]:
import pandas as pd
from src.config import RAW_DATA_PATH, RANDOM_SEED
from src.preprocessing import preprocess_meps
from src.features import add_features


In [229]:
df_raw = pd.read_excel(RAW_DATA_PATH)
df_pre = preprocess_meps(df_raw)
df_feat = add_features(df_pre)

In [230]:
TARGET_COLS = ["TOTEXPY2", "LOG_TOTEXPY2", "HIGHCOST_Y2", "ANY_ED_Y2", "ANY_IP_Y2"]

## Step 1) Define the modeling feature set (parsimonious)

In [231]:
cat_cols = [
    "RACE_ETH",
    "REGIONY1_CAT",
    "EDU_GROUP",
    "POVCATY1_CAT",
    "FAMSIZE_Y1_GRP",
    "INS_TYPE_Y1",
]

In [232]:
#Numeric (use engineered columns, not raw)

num_cols = [
    # demographics / SES
    "AGE",
    "SEX_BIN",
    "LOG_FAMINCY1",
    "FAMSIZE_Y1",

   

    # employment
    "WORKED_Y1",
    "ANY_UNEMP_COMP_Y1",
    "LOG_UNEMP_COMP_Y1",
    "EMP_INFO_R12",
    "EMP_ATTACHED_ANY_R12_FILL0",  # model-friendly version

    # health status baseline
    "RTHLTH1_FAIRPOOR",
    "MNHLTH1_FAIRPOOR",

    # chronic conditions baseline
    "HIBPDXY1_BIN",
    "CHDDXY1_BIN",
    "STRKDXY1_BIN",
    "CHOLDXY1_BIN",
    "ASTHDXY1_BIN",
    "DIABDXY1_M18_BIN",
    # "MULTIMORBIDITY_Y1",   # optional (can remove if you keep all *_BIN)

    # baseline utilisation/cost
    "LOG_TOTEXPY1",
    "ANY_ED_Y1",
    "ANY_IP_Y1",
]

In [265]:
FEATURES = cat_cols + num_cols

In [233]:
df_feat.head()

,DUID,PID,DUPERSID,PANEL,YEARIND,ALL5RDS,DIED,INST,MILITARY,ENTRSRVY,...,DIABDXY1_M18_BIN,MULTIMORBIDITY_Y1,MULTIMORBIDITY_GE2,LOG_TOTEXPY1,ANY_ED_Y1,ANY_IP_Y1,LOG_TOTEXPY2,HIGHCOST_Y2,ANY_ED_Y2,ANY_IP_Y2
0,2790002,101,2790002101,27,1,1,0,0,0,0,...,1,1,0,7.595890,0,0,6.472346,0,0,0
1,2790002,102,2790002102,27,1,1,0,0,0,0,...,0,1,0,0.000000,0,0,7.546974,0,0,0
2,2790004,101,2790004101,27,1,1,0,0,0,0,...,0,0,0,7.379632,0,0,6.894670,0,0,0
3,2790006,101,2790006101,27,1,1,0,0,0,0,...,1,3,1,7.374629,0,0,7.180070,0,0,0
4,2790006,102,2790006102,27,1,1,0,0,0,0,...,0,0,0,5.017280,0,0,0.000000,0,0,0


## Step 2) Add two “generic runners”

These mirror our baseline logic: same split, same preprocessing idea, validation-only hyperparam selection, and for classification threshold picked on validation by F1

In [236]:
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Dict, Any, Optional
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, root_mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, f1_score
)
from sklearn.model_selection import ParameterGrid
from sklearn.utils.class_weight import compute_sample_weight

from src.models import split_train_val_test, make_preprocess, best_threshold_by_f1 

@dataclass
class TuningResult:
    best_params: Dict[str, Any]
    valid_metrics: Dict[str, float]
    test_metrics: Dict[str, float]
    best_threshold: Optional[float] = None
    model: Optional[Pipeline] = None

def _trainval_concat(X_train, X_val, y_train, y_val):
    X_tv = pd.concat([X_train, X_val], axis=0)
    y_tv = pd.concat([y_train, y_val], axis=0)
    return X_tv, y_tv

def tune_regression_model(
    df: pd.DataFrame,
    target_col: str,
    num_cols: list[str],
    cat_cols: list[str],
    base_estimator,
    param_grid: list[dict],
    *,
    random_state: int = 42,
    scale_numeric: bool = False,   # trees: False; MLP: True
) -> TuningResult:
    X = df[num_cols + cat_cols].copy()
    y = df[target_col].copy()
    mask = y.notna()
    X, y = X.loc[mask], y.loc[mask]

    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        X, y, random_state=random_state, stratify=False
    )

    preprocess = make_preprocess(num_cols, cat_cols, scale_numeric=scale_numeric)

    best = None
    for params in param_grid:
        est = clone(base_estimator).set_params(**params)
        pipe = Pipeline([("preprocess", preprocess), ("model", est)])
        pipe.fit(X_train, y_train)

        val_pred = pipe.predict(X_val)
        val_metrics = {
            "RMSE_log": root_mean_squared_error(y_val, val_pred),
            "MAE_log": mean_absolute_error(y_val, val_pred),
            "R2": r2_score(y_val, val_pred),
        }

        score = val_metrics["R2"]  # choose by R2 (you can switch to -RMSE if preferred)
        if (best is None) or (score > best["score"]):
            best = {"score": score, "params": params, "val": val_metrics}

    # refit on train+val with best params, then test once
    X_tv, y_tv = _trainval_concat(X_train, X_val, y_train, y_val)
    best_est = clone(base_estimator).set_params(**best["params"])
    best_pipe = Pipeline([("preprocess", preprocess), ("model", best_est)])
    best_pipe.fit(X_tv, y_tv)

    test_pred = best_pipe.predict(X_test)
    test_metrics = {
        "RMSE_log": root_mean_squared_error(y_test, test_pred),
        "MAE_log": mean_absolute_error(y_test, test_pred),
        "R2": r2_score(y_test, test_pred),
    }

    return TuningResult(
        best_params=best["params"],
        valid_metrics=best["val"],
        test_metrics=test_metrics,
        model=best_pipe,
    )

def tune_classification_tree_model(
    df: pd.DataFrame,
    target_col: str,
    num_cols: list[str],
    cat_cols: list[str],
    base_estimator,
    param_grid: list[dict],
    *,
    random_state: int = 42,
    scale_numeric: bool = False,  # trees: usually False
) -> TuningResult:
    # ---------- 1. build X, y ----------
    X = df[num_cols + cat_cols].copy()
    y_raw = df[target_col].copy()

    mask = y_raw.notna()
    X = X.loc[mask]
    y = y_raw.loc[mask].astype(int)

    # ---------- 2. split ----------
    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        X, y, random_state=random_state, stratify=True
    )

    # ---------- 3. preprocessing ----------
    preprocess = make_preprocess(
        num_cols, cat_cols, scale_numeric=scale_numeric
    )

    # ---------- 4. hyperparameter tuning ----------
    best = None

    for params in param_grid:
        est = clone(base_estimator).set_params(**params)
        pipe = Pipeline([
            ("preprocess", preprocess),
            ("model", est),
        ])

        pipe.fit(X_train, y_train)

        val_proba = pipe.predict_proba(X_val)[:, 1]
        best_t, best_f1 = best_threshold_by_f1(y_val.values, val_proba)

        val_metrics = {
            "AUC": roc_auc_score(y_val, val_proba),
            "PR_AUC": average_precision_score(y_val, val_proba),
            "best_F1": best_f1,
            "best_t": best_t,
        }

        score = val_metrics["PR_AUC"]  # key metric for imbalance

        if (best is None) or (score > best["score"]):
            best = {
                "score": score,
                "params": params,
                "val": val_metrics,
            }

    # ---------- 5. refit on train + val ----------
    X_tv, y_tv = _trainval_concat(X_train, X_val, y_train, y_val)

    best_est = clone(base_estimator).set_params(**best["params"])
    best_pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", best_est),
    ])
    best_pipe.fit(X_tv, y_tv)

    # ---------- 6. final test evaluation ----------
    test_proba = best_pipe.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= best["val"]["best_t"]).astype(int)

    test_metrics = {
        "AUC": roc_auc_score(y_test, test_proba),
        "PR_AUC": average_precision_score(y_test, test_proba),
        "F1_at_best_t": f1_score(y_test, test_pred),
    }

    # ---------- 7. return ----------
    return TuningResult(
        best_params=best["params"],
        valid_metrics=best["val"],
        test_metrics=test_metrics,
        best_threshold=best["val"]["best_t"],
        model=best_pipe,
    )


## Step 3) Random Forest (regression + classification)

## Regression: LOG_TOTEXPY2

In [237]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Regression: LOG_TOTEXPY2
rf_reg = tune_regression_model(
    df_feat, target_col="LOG_TOTEXPY2",
    num_cols=num_cols, cat_cols=cat_cols,
    base_estimator=RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1),
    param_grid=[
        {"n_estimators": 400, "max_depth": None, "min_samples_leaf": 5},
        {"n_estimators": 600, "max_depth": 12,   "min_samples_leaf": 5},
        {"n_estimators": 600, "max_depth": 18,   "min_samples_leaf": 2},
    ],
    scale_numeric=False,
)


print("RF reg (test):", rf_reg.test_metrics)


RF reg (test): {'RMSE_log': 2.1785923532407176, 'MAE_log': 1.5450049079514576, 'R2': 0.5171668688778708}


调参

In [239]:
from sklearn.ensemble import RandomForestRegressor

rf_reg_tuned = tune_regression_model(
    df_feat, target_col="LOG_TOTEXPY2",
    num_cols=num_cols, cat_cols=cat_cols,
    base_estimator=RandomForestRegressor(
        random_state=RANDOM_SEED,
        n_jobs=-1,
        bootstrap=True,
    ),
    param_grid=[
        # try smaller leaves (less underfit)
        {"n_estimators": 1200, "max_depth": None, "min_samples_leaf": 1, "min_samples_split": 2,
         "max_features": 0.3, "max_samples": 0.8},
        {"n_estimators": 1200, "max_depth": 30,   "min_samples_leaf": 2, "min_samples_split": 5,
         "max_features": 0.3, "max_samples": 0.8},

        # sqrt often good baseline
        {"n_estimators": 800,  "max_depth": None, "min_samples_leaf": 2, "min_samples_split": 2,
         "max_features": "sqrt", "max_samples": 0.8},

        # stronger regularization
        {"n_estimators": 1500, "max_depth": 18,   "min_samples_leaf": 5, "min_samples_split": 10,
         "max_features": 0.2, "max_samples": 0.7},
    ],
    scale_numeric=False,
)
print(rf_reg_tuned.test_metrics)


{'RMSE_log': 2.152533438676481, 'MAE_log': 1.5332569293831346, 'R2': 0.5286484641140614}


## Classification example: HIGHCOST_Y2

In [240]:
# Classification example: HIGHCOST_Y2
rf_hc = tune_classification_tree_model(
    df_feat, target_col="HIGHCOST_Y2",
    num_cols=num_cols, cat_cols=cat_cols,
    base_estimator=RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample"),
    param_grid=[
        {"n_estimators": 500, "max_depth": None, "min_samples_leaf": 5},
        {"n_estimators": 800, "max_depth": 16,   "min_samples_leaf": 3},
    ],
    scale_numeric=False,
)

print("RF highcost (test):", rf_hc.test_metrics, "best_t:", rf_hc.best_threshold)


RF highcost (test): {'AUC': 0.8676443788384087, 'PR_AUC': 0.45889778426594363, 'F1_at_best_t': 0.5360230547550432} best_t: 0.5499999999999999


In [241]:
df_feat["HIGHCOST_Y2"].mean()


np.float64(0.10010240655401946)

prevalence is like 10% (common), then a random model PR-AUC ≈ 0.10. Getting 0.46 is a big improvement.

In [124]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# 1) rebuild X,y (same as in your tuning function)
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["HIGHCOST_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# 2) reproduce the same split (must match your tuning function)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

# 3) get test probabilities from the fitted pipeline you returned
proba_test = rf_hc.model.predict_proba(X_test)[:, 1]
best_t = rf_hc.best_threshold

# 4) compute confusion matrix / precision / recall
pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)

cm, prec, rec


(array([[1309,   98],
        [  63,   93]]),
 0.4869109947643979,
 0.5961538461538461)

High-cost classification (HIGHCOST_Y2).
The tuned Random Forest classifier shows strong discrimination for predicting Year-2 high-cost status. In the test set, it achieved ROC-AUC ≈ 0.87 and PR-AUC ≈ 0.46, substantially above the prevalence-based baseline (≈0.10). At the F1-optimal threshold (t ≈ 0.55), the model yields precision ≈ 0.49 and recall ≈ 0.60 (confusion matrix: TN=1309, FP=98, FN=63, TP=93), indicating that the flagged high-risk group has roughly five times the event rate of the overall population. Given the already strong performance, additional Random Forest hyperparameter tuning was deprioritized in favor of improving the more challenging utilization targets (ED and inpatient) and benchmarking against boosting models.

## Classification example: ANY_ED_Y2

In [125]:

# Classification example: ANY_ED_Y2
rf_ed = tune_classification_tree_model(
    df_feat, target_col="ANY_ED_Y2",
    num_cols=num_cols, cat_cols=cat_cols,
    base_estimator=RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample"),
    param_grid=[
        {"n_estimators": 500, "max_depth": None, "min_samples_leaf": 5},
        {"n_estimators": 800, "max_depth": 16,   "min_samples_leaf": 3},
    ],
    scale_numeric=False,
)

print("RF ED (test):", rf_ed.test_metrics, "best_t:", rf_ed.best_threshold)

RF ED (test): {'AUC': 0.7107891038083127, 'PR_AUC': 0.32531275523782766, 'F1_at_best_t': 0.3588516746411483} best_t: 0.49999999999999994


In [126]:
df_feat["ANY_ED_Y2"].mean()


np.float64(0.14272913466461853)

In [127]:
# 1) rebuild X,y (same as in your tuning function)
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["ANY_ED_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# 2) reproduce the same split (must match your tuning function)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

# 3) get test probabilities from the fitted pipeline you returned
proba_test = rf_ed.model.predict_proba(X_test)[:, 1]
best_t = rf_ed.best_threshold

# 4) compute confusion matrix / precision / recall
pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)

cm, prec, rec

(array([[1220,  120],
        [ 148,   75]]),
 0.38461538461538464,
 0.336322869955157)

## Classification example: ANY_IP_Y2

In [242]:

rf_ip = tune_classification_tree_model(
    df_feat, target_col="ANY_IP_Y2",
    num_cols=num_cols, cat_cols=cat_cols,
    base_estimator=RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample"),
    param_grid=[
        {"n_estimators": 500, "max_depth": None, "min_samples_leaf": 5},
        {"n_estimators": 800, "max_depth": 16,   "min_samples_leaf": 3},
    ],
    scale_numeric=False,
)

print("RF IP (test):", rf_ip.test_metrics, "best_t:", rf_ip.best_threshold)

RF IP (test): {'AUC': 0.7526823314006714, 'PR_AUC': 0.221199843214635, 'F1_at_best_t': 0.2641509433962264} best_t: 0.44999999999999996


In [243]:
df_feat["ANY_IP_Y2"].mean()

np.float64(0.07258064516129033)

In [244]:
# 1) rebuild X,y (same as in your tuning function)
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["ANY_IP_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# 2) reproduce the same split (must match your tuning function)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

# 3) get test probabilities from the fitted pipeline you returned
proba_test = rf_ip.model.predict_proba(X_test)[:, 1]
best_t = rf_ip.best_threshold

# 4) compute confusion matrix / precision / recall
pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)

cm, prec, rec

(array([[1333,  117],
        [  78,   35]]),
 0.23026315789473684,
 0.30973451327433627)

## 调参

最关键的通常是：

max_features（非常关键！减少树之间相关性，常常直接提升 AUC/PR-AUC）

max_samples（子采样，提升泛化、对不平衡也更稳）

min_samples_leaf（你可以尝试更小，避免欠拟合）

min_samples_split

分类继续用 class_weight="balanced_subsample" OK

In [245]:
rf_param_grid = [
    # 更强拟合 + 控制泛化
    {"n_estimators": 1200, "max_depth": None, "min_samples_leaf": 1, "min_samples_split": 2,
     "max_features": 0.3, "max_samples": 0.8, "bootstrap": True},

    {"n_estimators": 1200, "max_depth": 25, "min_samples_leaf": 2, "min_samples_split": 5,
     "max_features": 0.3, "max_samples": 0.8, "bootstrap": True},

    # sqrt 常见强基线
    {"n_estimators": 800, "max_depth": None, "min_samples_leaf": 2, "min_samples_split": 2,
     "max_features": "sqrt", "max_samples": 0.8, "bootstrap": True},

    # 更强正则
    {"n_estimators": 1500, "max_depth": 18, "min_samples_leaf": 5, "min_samples_split": 10,
     "max_features": 0.2, "max_samples": 0.7, "bootstrap": True},
]


In [136]:
from sklearn.ensemble import RandomForestClassifier

rf_ed_tuned = tune_classification_tree_model(
    df_feat, target_col="ANY_ED_Y2",
    num_cols=num_cols, cat_cols=cat_cols,
    base_estimator=RandomForestClassifier(
        random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample"
    ),
    param_grid=rf_param_grid,
    scale_numeric=False,
)
print("RF ed (test):", rf_ed_tuned.test_metrics, "best_t:", rf_ed_tuned.best_threshold)


RF ed (test): {'AUC': 0.7065390536108694, 'PR_AUC': 0.3204763544906881, 'F1_at_best_t': 0.3615819209039548} best_t: 0.39999999999999997


In [137]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# 1) rebuild X,y (same as in your tuning function)
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["ANY_ED_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# 2) reproduce the same split (must match your tuning function)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

# 3) get test probabilities from the fitted pipeline you returned
proba_test = rf_ed_tuned.model.predict_proba(X_test)[:, 1]
best_t = rf_ed_tuned.best_threshold

# 4) compute confusion matrix / precision / recall
pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)

cm, prec, rec

(array([[1128,  212],
        [ 127,   96]]),
 0.3116883116883117,
 0.4304932735426009)

1) ANY_ED_Y2（急诊）— 调参后是什么变化？
调参后（best_t=0.40）

AUC 0.7065，PR-AUC 0.3205，F1 0.3616

Confusion matrix [[1128, 212],[127, 96]]

Precision = 0.312

Recall = 0.430

预测为正的人数 = 212+96 = 308（≈19.7% test）

调参前（你之前 best_t≈0.50）

AUC 0.7108，PR-AUC 0.3253，F1 0.3589

Precision 0.385，Recall 0.336

预测为正人数 195（≈12.5% test）

解释（重点）

排序能力（AUC/PR-AUC）几乎没变，甚至略降 → 说明 RF 在 ED 任务上“天花板”就差不多在这了

这次调参带来的主要变化是 阈值/策略层面的 trade-off：

Recall 提升：0.336 → 0.430（抓到更多 ED）

Precision 下降：0.385 → 0.312（误报变多）

F1 小幅提升：0.359 → 0.362

✅ 从论文角度：你可以说 “RF 调参对 ED 的判别能力提升有限，但可通过阈值选择在 precision–recall 间权衡”。这非常合理。

A) 阈值表（t=0.2~0.8）

In [96]:
import numpy as np, pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

def threshold_table(y_true, proba, ths=np.arange(0.2, 0.81, 0.1)):
    rows = []
    for t in ths:
        pred = (proba >= t).astype(int)
        rows.append({
            "t": round(float(t), 2),
            "flagged_n": int(pred.sum()),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
        })
    return pd.DataFrame(rows)

# ED
ed_tbl = threshold_table(y_test.values, proba_test)   # 用你 ED 那次算出来的 y_test/proba_test
ed_tbl

,t,flagged_n,precision,recall,f1
0,0.2,1082,0.183919,0.892377,0.304981
1,0.3,615,0.227642,0.627803,0.334129
2,0.4,308,0.311688,0.430493,0.361582
3,0.5,165,0.406061,0.300448,0.345361
4,0.6,73,0.493151,0.161435,0.243243
5,0.7,21,0.523810,0.049327,0.090164
6,0.8,6,0.666667,0.017937,0.034934


B) Top-k 风险分层（比如 top 5% / 10% / 20%）

In [57]:
from sklearn.metrics import precision_score, recall_score

def topk_table(y_true, proba, fracs=(0.05, 0.10, 0.20)):
    n = len(proba)
    order = np.argsort(-proba)
    rows = []
    for frac in fracs:
        k = int(round(frac * n))
        pred = np.zeros(n, dtype=int)
        pred[order[:k]] = 1
        rows.append({
            "top_frac": frac,
            "flagged_n": k,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
        })
    return pd.DataFrame(rows)

topk_table(y_test.values, proba_test)


,top_frac,flagged_n,precision,recall
0,0.05,78,0.487179,0.170404
1,0.10,156,0.410256,0.286996
2,0.20,313,0.313099,0.439462


In [246]:
from sklearn.ensemble import RandomForestClassifier

rf_ip_tuned = tune_classification_tree_model(
    df_feat, target_col="ANY_IP_Y2",
    num_cols=num_cols, cat_cols=cat_cols,
    base_estimator=RandomForestClassifier(
        random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample"
    ),
    param_grid=rf_param_grid,
    scale_numeric=False,
)
print("RF ip (test):", rf_ip_tuned.test_metrics, "best_t:", rf_ip_tuned.best_threshold)

RF ip (test): {'AUC': 0.7545315837656394, 'PR_AUC': 0.22548500746386185, 'F1_at_best_t': 0.2847222222222222} best_t: 0.39999999999999997


In [139]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# 1) rebuild X,y (same as in your tuning function)
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["ANY_IP_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# 2) reproduce the same split (must match your tuning function)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

# 3) get test probabilities from the fitted pipeline you returned
proba_test = rf_ip_tuned.model.predict_proba(X_test)[:, 1]
best_t = rf_ip_tuned.best_threshold

# 4) compute confusion matrix / precision / recall
pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)

cm, prec, rec

(array([[1316,  134],
        [  72,   41]]),
 0.2342857142857143,
 0.36283185840707965)

2) ANY_IP_Y2（住院）— 调参后是明确提升
调参后（best_t=0.40）

AUC 0.7545，PR-AUC 0.2255，F1 0.2847

Confusion matrix [[1316, 134],[72, 41]]

Precision = 0.234

Recall = 0.363

预测为正人数 = 134+41 = 175（≈11.2% test）

调参前（best_t≈0.45）

AUC 0.7527，PR-AUC 0.2212，F1 0.2642

Precision 0.230，Recall 0.310

预测为正人数 152

✅ 这里是一致提升：

AUC/PR-AUC ↑（判别能力提升）

Recall ↑（抓到更多住院）

Precision 基本持平

F1 ↑（0.264 → 0.285）

一件“论文加分”的事

把这两任务都做一个 阈值表 / top-k 表（不用换模型就能让结果更“决策友好”）：

ED：展示 t=0.3/0.4/0.5 下的 precision/recall/flagged_n（你现在等于从 0.5 走到 0.4）

IP：同理，强调在固定资源下（比如 top 10% 风险）能抓到多少住院

A) 阈值表（t=0.2~0.8）

In [99]:

import numpy as np, pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

def threshold_table(y_true, proba, ths=np.arange(0.2, 0.81, 0.1)):
    rows = []
    for t in ths:
        pred = (proba >= t).astype(int)
        rows.append({
            "t": round(float(t), 2),
            "flagged_n": int(pred.sum()),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
        })
    return pd.DataFrame(rows)

# IP
ip_tbl = threshold_table(y_test.values, proba_test)   
ip_tbl

,t,flagged_n,precision,recall,f1
0,0.2,550,0.145455,0.707965,0.241327
1,0.3,331,0.175227,0.513274,0.261261
2,0.4,175,0.234286,0.362832,0.284722
3,0.5,88,0.318182,0.247788,0.278607
4,0.6,38,0.394737,0.132743,0.198675
5,0.7,15,0.333333,0.044248,0.078125
6,0.8,4,0.500000,0.017699,0.034188


B) Top-k 风险分层（比如 top 5% / 10% / 20%）

In [100]:
from sklearn.metrics import precision_score, recall_score

def topk_table(y_true, proba, fracs=(0.05, 0.10, 0.20)):
    n = len(proba)
    order = np.argsort(-proba)
    rows = []
    for frac in fracs:
        k = int(round(frac * n))
        pred = np.zeros(n, dtype=int)
        pred[order[:k]] = 1
        rows.append({
            "top_frac": frac,
            "flagged_n": k,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
        })
    return pd.DataFrame(rows)

topk_table(y_test.values, proba_test)


,top_frac,flagged_n,precision,recall
0,0.05,78,0.333333,0.230088
1,0.10,156,0.237179,0.327434
2,0.20,313,0.182109,0.504425


1) proba 是什么？怎么理解？

proba 就是模型对每个人输出的 “发生事件(=1) 的预测概率/风险分数”：

代码里它来自：predict_proba(X_test)[:, 1]

对 RandomForest 来说，它大致等于：投票为 1 的树的比例
例如 proba=0.40 ≈ 40% 的树认为会发生事件（ED 或 IP）

⚠️ 重要：RF 的 proba 不一定是严格校准的真实概率（可能偏高/偏低），但它通常能很好地用于：

排序（谁更高风险）

做阈值决策（proba≥t 判为 1）

Top-k 风险分层（挑风险最高的前 5%/10%/20%）

2) ED（ANY_ED_Y2）怎么看？
A) 阈值表（t=0.2~0.8）

你这张表很清晰地展示了 trade-off：

t=0.2：flagged 1082 人，precision 0.184，recall 0.892
→ 很像“筛查”：几乎都抓到了，但误报很多

t=0.4（你当前 best_t）：flagged 308 人，precision 0.312，recall 0.430，F1=0.362（最高）
→ 最“平衡”的点

t=0.5：flagged 165 人，precision 0.406，recall 0.300
→ 更“精准”，但漏掉更多

一句话总结 ED：

如果你在论文里用“最大 F1”的规则，t≈0.4 是合理的；如果你更在意“不要漏掉”，就把阈值降到 0.3 或 0.2；如果更在意“少误报”，阈值升到 0.5/0.6。

B) Top-k 风险分层（你第二张图）

top 5%：precision 0.487，recall 0.170

top 10%：precision 0.410，recall 0.287

top 20%：precision 0.313，recall 0.439

这特别有意思：top 20% 的 precision/recall 跟阈值 t≈0.4 几乎一样（因为 t=0.4 时 flagged 308，人群规模和 top20% ≈313 很接近）。
所以你可以在论文里说：

“Using either a probability threshold around 0.4 or selecting the top 20% highest-risk individuals yields similar operating characteristics.”

3) IP（ANY_IP_Y2）怎么看？
A) 阈值表

你这里的 最佳 F1 也在 t=0.4（F1≈0.285），很一致：

t=0.2：flagged 550，precision 0.145，recall 0.708
→ 典型“筛查模式”：抓很多，但误报也多

t=0.4（best）：flagged 175，precision 0.234，recall 0.363，F1=0.285
→ 平衡点

t=0.5：flagged 88，precision 0.318，recall 0.248
→ 更精准但漏更多

B) Top-k（你第三张图）

top 5%：precision 0.333，recall 0.230

top 10%：precision 0.237，recall 0.327

top 20%：precision 0.182，recall 0.504

这里也能看出：top 20% 的 recall 0.50 很接近阈值 t≈0.3 的 recall 0.51（因为 t=0.3 flagged 331，规模更接近 top20% 的 313）。

Using a tuned Random Forest classifier, the model achieved moderate discrimination for predicting Year-2 ED use (AUC ≈ 0.71; PR-AUC ≈ 0.32). Operational performance depends strongly on the decision threshold. When selecting the threshold that maximizes F1, the optimal cutoff is around t = 0.40, flagging 308 individuals in the test set and yielding precision ≈ 0.31, recall ≈ 0.43, and F1 ≈ 0.36. Lower thresholds prioritize sensitivity (e.g., t = 0.20 reaches recall ≈ 0.89 but with low precision), whereas higher thresholds prioritize precision (e.g., t = 0.60 yields precision ≈ 0.49 but recall ≈ 0.16).
As an alternative, risk stratification based on predicted probabilities provides an interpretable resource-constrained policy: selecting the top 10% highest-risk individuals results in precision ≈ 0.41 and recall ≈ 0.29, while selecting the top 20% yields precision ≈ 0.31 and recall ≈ 0.44, closely matching the t ≈ 0.40 operating point.

For Year-2 inpatient admission (ANY_IP_Y2), the tuned Random Forest shows slightly stronger ranking performance than for ED (AUC ≈ 0.75; PR-AUC ≈ 0.23). The F1-optimal decision threshold is again around t = 0.40, flagging 175 individuals and yielding precision ≈ 0.23, recall ≈ 0.36, and F1 ≈ 0.28. As expected for a rarer outcome, precision increases as the threshold rises at the cost of recall (e.g., t = 0.50 gives precision ≈ 0.32 with recall ≈ 0.25), while lower thresholds improve recall substantially (e.g., t = 0.20 yields recall ≈ 0.71 but precision ≈ 0.15).
Under a resource-constrained risk stratification strategy, selecting the top 5% highest-risk individuals achieves precision ≈ 0.33 and recall ≈ 0.23; selecting the top 10% yields precision ≈ 0.24 and recall ≈ 0.33, and the top 20% captures about half of admissions (recall ≈ 0.50) with precision ≈ 0.18.

Overall, these results illustrate that threshold choice (or a top-k policy) translates the same ranking model into different operational trade-offs, enabling either screening-oriented (high recall) or resource-targeted (higher precision) interventions.

# Step 4) XGBoost

In [63]:
from xgboost import XGBClassifier, XGBRegressor


## regression  LOG_TOTEXPY2

In [247]:
# Regression
xgb_reg = tune_regression_model(
    df_feat, target_col="LOG_TOTEXPY2",
    num_cols=num_cols, cat_cols=cat_cols,
    base_estimator=XGBRegressor(
        random_state=RANDOM_SEED,
        tree_method="hist",
        n_jobs=-1,
        eval_metric="rmse",
        ),
        param_grid=[
        {"n_estimators": 800, "max_depth": 4, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8},
        {"n_estimators": 1200,"max_depth": 3, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8},
        ],
        scale_numeric=False,
)


print("XGB reg (test):", xgb_reg.test_metrics)

XGB reg (test): {'RMSE_log': 2.1586774041740204, 'MAE_log': 1.5340073566000356, 'R2': 0.5259538713198084}


In [248]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1) split
X = df_feat[num_cols + cat_cols].copy()
y = df_feat["LOG_TOTEXPY2"].copy()
mask = y.notna()
X, y = X.loc[mask], y.loc[mask]

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=False
)

# 2) preprocess -> arrays (one-hot etc.)
pre = make_preprocess(num_cols, cat_cols, scale_numeric=False)
X_train_p = pre.fit_transform(X_train)
X_val_p   = pre.transform(X_val)
X_test_p  = pre.transform(X_test)

# 3) DMatrix
dtrain = xgb.DMatrix(X_train_p, label=y_train)
dval   = xgb.DMatrix(X_val_p,   label=y_val)
dtest  = xgb.DMatrix(X_test_p,  label=y_test)

# 4) candidate params (use xgb.train param names: eta/lambda/alpha)
cands = [
    # {"max_depth": 3, "eta": 0.03, "subsample": 0.9, "colsample_bytree": 0.9,
    #  "min_child_weight": 5, "gamma": 0.0, "lambda": 2.0, "alpha": 0.0},
    {"max_depth": 3, "eta": 0.03, "subsample": 0.8, "colsample_bytree": 0.8,
    "min_child_weight": 10, "gamma": 0.5, "lambda": 5.0, "alpha": 0.0},

    {"max_depth": 3, "eta": 0.02, "subsample": 0.8, "colsample_bytree": 0.8,
     "min_child_weight": 10, "gamma": 0.5, "lambda": 5.0, "alpha": 0.0},
    # {"max_depth": 4, "eta": 0.03, "subsample": 0.9, "colsample_bytree": 0.9,
    # "min_child_weight": 5, "gamma": 0.0, "lambda": 2.0, "alpha": 0.0},

]

base_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "seed": RANDOM_SEED,
}

best = None

for p in cands:
    params = dict(base_params)
    params.update(p)

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=5000,
        evals=[(dval, "val")],
        early_stopping_rounds=200,
        verbose_eval=False
    )

    # best iteration handling across versions
    best_iter = getattr(booster, "best_iteration", None)
    best_ntree_limit = getattr(booster, "best_ntree_limit", None)

    if best_iter is not None:
        val_pred = booster.predict(dval, iteration_range=(0, best_iter + 1))
        val_rmse = root_mean_squared_error(y_val, val_pred)
    else:
        # older versions use ntree_limit
        val_pred = booster.predict(dval, ntree_limit=best_ntree_limit)
        val_rmse = root_mean_squared_error(y_val, val_pred)

    if (best is None) or (val_rmse < best["val_rmse"]):
        best = {
            "val_rmse": val_rmse,
            "booster": booster,
            "params": params,
            "best_iter": best_iter,
            "best_ntree_limit": best_ntree_limit
        }

# 5) test eval with best booster
booster = best["booster"]
best_iter = best["best_iter"]
best_ntree_limit = best["best_ntree_limit"]

if best_iter is not None:
    test_pred = booster.predict(dtest, iteration_range=(0, best_iter + 1))
else:
    test_pred = booster.predict(dtest, ntree_limit=best_ntree_limit)

rmse = root_mean_squared_error(y_test, test_pred)
mae  = mean_absolute_error(y_test, test_pred)
r2   = r2_score(y_test, test_pred)

print("xgboost version:", xgb.__version__)
print("best params:", best["params"])
print("best_iteration:", best_iter if best_iter is not None else best_ntree_limit)
print({"RMSE_log": rmse, "MAE_log": mae, "R2": r2})


xgboost version: 3.1.2
best params: {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'seed': 42, 'max_depth': 3, 'eta': 0.02, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 10, 'gamma': 0.5, 'lambda': 5.0, 'alpha': 0.0}
best_iteration: 1037
{'RMSE_log': 2.1500502264902157, 'MAE_log': 1.5415488342366697, 'R2': 0.5297353608086273}


“For LOG_TOTEXPY2, Random Forest and XGBoost achieved very similar performance. After enabling early stopping and stronger regularization, XGBoost slightly improved RMSE and R² compared to the tuned Random Forest, while Random Forest retained a marginal advantage in MAE. Overall, the results suggest diminishing returns from increasing model complexity once expenditures are log-transformed.”

## classification HIGHCOST_Y2

In [67]:
# Classification 
xgb_hc = tune_classification_tree_model(
        df_feat, target_col="HIGHCOST_Y2",
        num_cols=num_cols, cat_cols=cat_cols,
        base_estimator=XGBClassifier(
            random_state=RANDOM_SEED,
            tree_method="hist",
            n_jobs=-1,
            eval_metric="logloss",
            
        ),
        param_grid=[
            {"n_estimators": 800, "max_depth": 3, "learning_rate": 0.05, "subsample": 0.9, "colsample_bytree": 0.9},
            {"n_estimators": 1200,"max_depth": 3, "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 0.9},
        ],
        scale_numeric=False,
    )

print("XGB highcost (test):", xgb_hc.test_metrics, "best_t:", xgb_hc.best_threshold)

XGB highcost (test): {'AUC': 0.8550106609808102, 'PR_AUC': 0.4339340037410966, 'F1_at_best_t': 0.5292479108635098} best_t: 0.25


In [68]:
# rebuild X,y for HIGHCOST
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["HIGHCOST_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# split (same function + same seed as your tuning)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

pos = y_train.sum()
neg = len(y_train) - pos
spw = neg / pos

print("train prevalence:", float(pos/len(y_train)))
print("scale_pos_weight:", float(spw))


train prevalence: 0.10008536064874093
scale_pos_weight: 8.991471215351812


In [70]:
from xgboost import XGBClassifier

xgb_base = XGBClassifier(
    random_state=RANDOM_SEED,
    tree_method="hist",
    n_jobs=-1,
    eval_metric="logloss",
    scale_pos_weight=spw,   # ✅关键
)

xgb_hc = tune_classification_tree_model(
    df_feat, target_col="HIGHCOST_Y2",
    num_cols=num_cols, cat_cols=cat_cols,
    base_estimator=xgb_base,
    param_grid=[
        {"n_estimators": 2000, "max_depth": 3, "learning_rate": 0.02, "subsample": 0.9, "colsample_bytree": 0.9,
         "min_child_weight": 5, "reg_lambda": 1.0, "gamma": 0.0},
        {"n_estimators": 1500, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 0.9,
         "min_child_weight": 5, "reg_lambda": 2.0, "gamma": 0.0},
        {"n_estimators": 1500, "max_depth": 3, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8,
         "min_child_weight": 10, "reg_lambda": 5.0, "gamma": 0.5},
    ],
    scale_numeric=False,
)
print("XGB highcost (test):", xgb_hc.test_metrics, "best_t:", xgb_hc.best_threshold)


XGB highcost (test): {'AUC': 0.8555573779454376, 'PR_AUC': 0.44329432701217614, 'F1_at_best_t': 0.5249343832020997} best_t: 0.65


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# 1) rebuild X,y (same as in your tuning function)
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["HIGHCOST_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# 2) reproduce the same split (must match your tuning function)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

# 3) get test probabilities from the fitted pipeline you returned
proba_test = xgb_hc.model.predict_proba(X_test)[:, 1]
best_t = xgb_hc.best_threshold

# 4) compute confusion matrix / precision / recall
pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)

cm, prec, rec

(array([[1282,  125],
        [  56,  100]]),
 0.4444444444444444,
 0.6410256410256411)

“For HIGHCOST_Y2, Random Forest slightly outperformed XGBoost in both ROC-AUC and PR-AUC, indicating better ranking performance. XGBoost achieved higher recall at its F1-optimal threshold but at the cost of reduced precision; overall F1 remained marginally lower than Random Forest.”

## classification ANY_ED_Y2

In [101]:

xgb_ed = tune_classification_tree_model(
        df_feat, target_col="ANY_ED_Y2",
        num_cols=num_cols, cat_cols=cat_cols,
        base_estimator=XGBClassifier(
            random_state=RANDOM_SEED,
            tree_method="hist",
            n_jobs=-1,
            eval_metric="logloss",
            
        ),
        param_grid=[
            {"n_estimators": 800, "max_depth": 3, "learning_rate": 0.05, "subsample": 0.9, "colsample_bytree": 0.9},
            {"n_estimators": 1200,"max_depth": 3, "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 0.9},
        ],
        scale_numeric=False,
    )

print("XGB ed (test):", xgb_ed.test_metrics, "best_t:", xgb_ed.best_threshold)

XGB ed (test): {'AUC': 0.7278595810186734, 'PR_AUC': 0.34953461893123483, 'F1_at_best_t': 0.3656716417910448} best_t: 0.2


In [102]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# 1) rebuild X,y (same as in your tuning function)
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["ANY_ED_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# 2) reproduce the same split (must match your tuning function)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

# 3) get test probabilities from the fitted pipeline you returned
proba_test = xgb_ed.model.predict_proba(X_test)[:, 1]
best_t = xgb_ed.best_threshold

# 4) compute confusion matrix / precision / recall
pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)

cm, prec, rec

(array([[1125,  215],
        [ 125,   98]]),
 0.31309904153354634,
 0.43946188340807174)

“For ANY_ED_Y2, XGBoost modestly outperformed the tuned Random Forest (ROC-AUC ≈ 0.73 vs 0.71; PR-AUC ≈ 0.35 vs 0.32), indicating improved ranking of ED risk. At the F1-optimal threshold, both models produced similar precision–recall trade-offs.”

## classification ANY_IP_Y2

In [103]:

xgb_ip = tune_classification_tree_model(
        df_feat, target_col="ANY_IP_Y2",
        num_cols=num_cols, cat_cols=cat_cols,
        base_estimator=XGBClassifier(
            random_state=RANDOM_SEED,
            tree_method="hist",
            n_jobs=-1,
            eval_metric="logloss",
            
        ),
        param_grid=[
            {"n_estimators": 800, "max_depth": 3, "learning_rate": 0.05, "subsample": 0.9, "colsample_bytree": 0.9},
            {"n_estimators": 1200,"max_depth": 3, "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 0.9},
        ],
        scale_numeric=False,
    )

print("XGB ip (test):", xgb_ip.test_metrics, "best_t:", xgb_ip.best_threshold)

XGB ip (test): {'AUC': 0.7522978333841929, 'PR_AUC': 0.2020236656065295, 'F1_at_best_t': 0.2611464968152866} best_t: 0.15


In [104]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# 1) rebuild X,y (same as in your tuning function)
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["ANY_IP_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# 2) reproduce the same split (must match your tuning function)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

# 3) get test probabilities from the fitted pipeline you returned
proba_test = xgb_ip.model.predict_proba(X_test)[:, 1]
best_t = xgb_ip.best_threshold

# 4) compute confusion matrix / precision / recall
pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)

cm, prec, rec

(array([[1290,  160],
        [  72,   41]]),
 0.20398009950248755,
 0.36283185840707965)

论文层面，你可以很自然地做这样的结论：

ED：XGB > RF（你已经看到 PR-AUC 提升）

IP：RF > XGB（你现在看到 PR-AUC 和 F1 都更高）

Highcost：RF > XGB（也更好）

回归：XGB ~ RF（几乎打平）

这反而是个很好的 thesis story：不同目标适合不同模型，并不是 boosting 永远赢。

# MLP

## regression 

In [105]:
from sklearn.neural_network import MLPRegressor

mlp_reg = tune_regression_model(
    df_feat, target_col="LOG_TOTEXPY2",
    num_cols=num_cols, cat_cols=cat_cols,
    base_estimator=MLPRegressor(random_state=RANDOM_SEED, early_stopping=True),
    param_grid=[
        {"hidden_layer_sizes": (128,64), "alpha": 1e-4, "max_iter": 400, "learning_rate_init": 1e-3},
        {"hidden_layer_sizes": (256,128), "alpha": 1e-4, "max_iter": 400, "learning_rate_init": 5e-4},
    ],
    scale_numeric=True,   # ✅ MLP 一定要 True
)
print(mlp_reg.test_metrics)


{'RMSE_log': 2.1494636826311613, 'MAE_log': 1.5472050608155734, 'R2': 0.5299919066407873}


## classification

你之前的 make_preprocess 不能直接给 MLP 用，原因通常是这两个：

输出是稀疏矩阵 (sparse)（OneHotEncoder 默认 sparse）

MLPClassifier/Regressor 在很多 sklearn 版本里对 sparse 支持不好/会很慢/直接报错

没做标准化 (StandardScaler)

MLP 对特征尺度非常敏感，不 scale 往往效果差、收敛慢

In [106]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight


def _make_preprocess_for_mlp(num_cols, cat_cols):
    """Dense output preprocessing for MLP: impute + scale + one-hot (dense)."""
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    # sklearn 1.2+ uses sparse_output; older uses sparse
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", ohe),
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    return pre


def tune_classification_mlp_model(
    df: pd.DataFrame,
    target_col: str,
    num_cols: list[str],
    cat_cols: list[str],
    param_grid: list[dict],
    *,
    random_state: int = 42,
) -> "TuningResult":
    """
    Tune MLPClassifier on (train, val), select best params by val PR-AUC,
    choose threshold by val best F1, refit on train+val, evaluate on test.
    Uses balanced sample_weight automatically.
    """

    # ---------- 1) build X,y ----------
    X = df[num_cols + cat_cols].copy()
    y_raw = df[target_col].copy()
    mask = y_raw.notna()
    X = X.loc[mask]
    y = y_raw.loc[mask].astype(int)

    # ---------- 2) split ----------
    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        X, y, random_state=random_state, stratify=True
    )

    # ---------- 3) preprocess (dense) ----------
    preprocess = _make_preprocess_for_mlp(num_cols, cat_cols)

    # ---------- 4) tuning ----------
    best = None

    for params in param_grid:
        base = MLPClassifier(
            random_state=random_state,
            early_stopping=True,
            max_iter=400,
        )
        est = clone(base).set_params(**params)

        pipe = Pipeline([
            ("preprocess", preprocess),
            ("model", est),
        ])

        # balanced sample weights on TRAIN
        sw_train = compute_sample_weight(class_weight="balanced", y=y_train)

        # MLPClassifier.fit supports sample_weight
        pipe.fit(X_train, y_train, model__sample_weight=sw_train)

        val_proba = pipe.predict_proba(X_val)[:, 1]
        best_t, best_f1 = best_threshold_by_f1(y_val.values, val_proba)

        val_metrics = {
            "AUC": roc_auc_score(y_val, val_proba),
            "PR_AUC": average_precision_score(y_val, val_proba),
            "best_F1": best_f1,
            "best_t": best_t,
        }

        score = val_metrics["PR_AUC"]  # imbalance-friendly

        if (best is None) or (score > best["score"]):
            best = {"score": score, "params": params, "val": val_metrics, "pipe": pipe}

    # ---------- 5) refit on train + val ----------
    X_tv, y_tv = _trainval_concat(X_train, X_val, y_train, y_val)

    base = MLPClassifier(
        random_state=random_state,
        early_stopping=True,
        max_iter=400,
    )
    best_est = clone(base).set_params(**best["params"])
    best_pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", best_est),
    ])

    sw_tv = compute_sample_weight(class_weight="balanced", y=y_tv)
    best_pipe.fit(X_tv, y_tv, model__sample_weight=sw_tv)

    # ---------- 6) test ----------
    test_proba = best_pipe.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= best["val"]["best_t"]).astype(int)

    test_metrics = {
        "AUC": roc_auc_score(y_test, test_proba),
        "PR_AUC": average_precision_score(y_test, test_proba),
        "F1_at_best_t": f1_score(y_test, test_pred),
    }

    return TuningResult(
        best_params=best["params"],
        valid_metrics=best["val"],
        test_metrics=test_metrics,
        best_threshold=best["val"]["best_t"],
        model=best_pipe,
    )


## classification HIGHCOST_Y2

In [109]:
mlp_hc = tune_classification_mlp_model(
    df_feat,
    target_col="HIGHCOST_Y2",
    num_cols=num_cols,
    cat_cols=cat_cols,
    param_grid=[
        {"hidden_layer_sizes": (128, 64), "alpha": 1e-4, "learning_rate_init": 1e-3},
        {"hidden_layer_sizes": (256, 128), "alpha": 1e-4, "learning_rate_init": 5e-4},
        {"hidden_layer_sizes": (128, 64), "alpha": 1e-3, "learning_rate_init": 1e-3},
    ],
    random_state=RANDOM_SEED,
)

print("mlp high cost:", mlp_hc.test_metrics, "best_t:", mlp_hc.best_threshold)

mlp high cost: {'AUC': 0.8530151440599201, 'PR_AUC': 0.4204354885641735, 'F1_at_best_t': 0.4594594594594595} best_t: 0.75


In [110]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# 1) rebuild X,y (same as in your tuning function)
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["HIGHCOST_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# 2) reproduce the same split (must match your tuning function)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

# 3) get test probabilities from the fitted pipeline you returned
proba_test = mlp_hc.model.predict_proba(X_test)[:, 1]
best_t = mlp_hc.best_threshold

# 4) compute confusion matrix / precision / recall
pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)

cm, prec, rec

(array([[1278,  129],
        [  71,   85]]),
 0.397196261682243,
 0.5448717948717948)

In [112]:
mlp_ed = tune_classification_mlp_model(
    df_feat,
    target_col="ANY_ED_Y2",
    num_cols=num_cols,
    cat_cols=cat_cols,
    param_grid=[
        {"hidden_layer_sizes": (128, 64), "alpha": 1e-4, "learning_rate_init": 1e-3},
        {"hidden_layer_sizes": (256, 128), "alpha": 1e-4, "learning_rate_init": 5e-4},
        {"hidden_layer_sizes": (128, 64), "alpha": 1e-3, "learning_rate_init": 1e-3},
    ],
    random_state=RANDOM_SEED,
)

print("mlp ed:", mlp_ed.test_metrics, "best_t:", mlp_ed.best_threshold)


mlp ed: {'AUC': 0.710180041496553, 'PR_AUC': 0.32818693375833213, 'F1_at_best_t': 0.3436928702010969} best_t: 0.6


In [113]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# 1) rebuild X,y (same as in your tuning function)
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["ANY_ED_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# 2) reproduce the same split (must match your tuning function)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

# 3) get test probabilities from the fitted pipeline you returned
proba_test = mlp_ed.model.predict_proba(X_test)[:, 1]
best_t = mlp_ed.best_threshold

# 4) compute confusion matrix / precision / recall
pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)

cm, prec, rec

(array([[1110,  230],
        [ 129,   94]]),
 0.29012345679012347,
 0.42152466367713004)

In [114]:
mlp_ip = tune_classification_mlp_model(
    df_feat,
    target_col="ANY_IP_Y2",
    num_cols=num_cols,
    cat_cols=cat_cols,
    param_grid=[
        {"hidden_layer_sizes": (128, 64), "alpha": 1e-4, "learning_rate_init": 1e-3},
        {"hidden_layer_sizes": (256, 128), "alpha": 1e-4, "learning_rate_init": 5e-4},
        {"hidden_layer_sizes": (128, 64), "alpha": 1e-3, "learning_rate_init": 1e-3},
    ],
    random_state=RANDOM_SEED,
)

print("mlp ip:", mlp_ip.test_metrics, "best_t:", mlp_ip.best_threshold)

mlp ip: {'AUC': 0.7580408910588953, 'PR_AUC': 0.20154267281054764, 'F1_at_best_t': 0.2747603833865815} best_t: 0.65


In [116]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# 1) rebuild X,y (same as in your tuning function)
X = df_feat[num_cols + cat_cols].copy()
y_raw = df_feat["ANY_IP_Y2"].copy()

mask = y_raw.notna()
X = X.loc[mask]
y = y_raw.loc[mask].astype(int)

# 2) reproduce the same split (must match your tuning function)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=True
)

# 3) get test probabilities from the fitted pipeline you returned
proba_test = mlp_ip.model.predict_proba(X_test)[:, 1]
best_t = mlp_ip.best_threshold

# 4) compute confusion matrix / precision / recall
pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)

cm, prec, rec

(array([[1293,  157],
        [  70,   43]]),
 0.215,
 0.3805309734513274)

# Save the preferred pipelines
## For RF / XGBClassifier / MLP (sklearn Pipelines)

In [255]:
from pathlib import Path
import joblib
import json
import sklearn
import xgboost
import datetime as dt

ART_DIR = Path("results/model_artifacts")
ART_DIR.mkdir(parents=True, exist_ok=True)

def save_sklearn_artifact(name: str, pipeline, meta: dict):
    path = ART_DIR / f"{name}.joblib"
    joblib.dump(pipeline, path)
    meta_path = ART_DIR / f"{name}.meta.json"
    meta = {
        **meta,
        "saved_at": dt.datetime.now().isoformat(),
        "sklearn_version": sklearn.__version__,
        "xgboost_version": getattr(xgboost, "__version__", None),
    }
    meta_path.write_text(json.dumps(meta, indent=2))
    return str(path), str(meta_path)


## save selected classification models

In [256]:
# Example mapping (use YOUR chosen ones)

save_sklearn_artifact(
    "clf_highcost_rf",
    rf_hc.model,
    meta={"target":"HIGHCOST_Y2", "model":"RandomForest", "best_threshold": rf_hc.best_threshold, "test_metrics": rf_hc.test_metrics}
)

save_sklearn_artifact(
    "clf_ed_xgb",
    xgb_ed.model,
    meta={"target":"ANY_ED_Y2", "model":"XGBoost", "best_threshold": xgb_ed.best_threshold, "test_metrics": xgb_ed.test_metrics}
)

save_sklearn_artifact(
    "clf_ip_rf",
    rf_ip_tuned.model,
    meta={"target":"ANY_IP_Y2", "model":"RandomForest_tuned", "best_threshold": rf_ip_tuned.best_threshold, "test_metrics": rf_ip_tuned.test_metrics}
)


('results/model_artifacts/clf_ip_rf.joblib',
 'results/model_artifacts/clf_ip_rf.meta.json')

In [251]:
import sys, sklearn
print(sys.executable)
print("sklearn", sklearn.__version__)


/opt/anaconda3/envs/meps/bin/python
sklearn 1.7.2


## load

In [ ]:
# pipe = joblib.load(ART_DIR / "clf_highcost_rf.joblib")
# meta = json.loads((ART_DIR / "clf_highcost_rf.meta.json").read_text())
# t = meta["best_threshold"]

# pipe = joblib.load(ART_DIR / "clf_highcost_rf.joblib")
# meta = json.loads((ART_DIR / "clf_highcost_rf.meta.json").read_text())
# t = meta["best_threshold"]

# proba = pipe.predict_proba(X_new)[:, 1]
# pred = (proba >= t).astype(int)



Special case: your XGB regression with early stopping (xgb.train)

Because your environment’s sklearn wrapper doesn’t support early stopping, you trained the best regressor with xgboost.train + Booster. For this case, save:

the preprocessor (joblib)

the Booster (save_model)

a small meta json including best_iteration

In [257]:
import joblib, json
from pathlib import Path
import datetime as dt
import sklearn, xgboost

def save_xgb_booster_reg(name: str, preprocess, booster, meta: dict):
    pre_path = ART_DIR / f"{name}.preprocess.joblib"
    model_path = ART_DIR / f"{name}.booster.json"
    meta_path = ART_DIR / f"{name}.meta.json"

    joblib.dump(preprocess, pre_path)
    booster.save_model(model_path)

    meta = {
        **meta,
        "saved_at": dt.datetime.now().isoformat(),
        "sklearn_version": sklearn.__version__,
        "xgboost_version": xgboost.__version__,
    }
    meta_path.write_text(json.dumps(meta, indent=2))
    return str(pre_path), str(model_path), str(meta_path)


In [258]:
# suppose your best dict is: best["booster"], preprocess=pre, best_iter=best_iter
save_xgb_booster_reg(
    "reg_log_totexpy2_xgb_es",
    preprocess=pre,
    booster=best["booster"],
    meta={
        "target":"LOG_TOTEXPY2",
        "model":"xgboost.train",
        "best_iteration": best_iter,
        "test_metrics": {"RMSE_log": rmse, "MAE_log": mae, "R2": r2}
    }
)


('results/model_artifacts/reg_log_totexpy2_xgb_es.preprocess.joblib',
 'results/model_artifacts/reg_log_totexpy2_xgb_es.booster.json',
 'results/model_artifacts/reg_log_totexpy2_xgb_es.meta.json')

to predict later

In [ ]:
# import xgboost as xgb
# import joblib, json
# from pathlib import Path

# pre = joblib.load(ART_DIR / "reg_log_totexpy2_xgb_es.preprocess.joblib")
# booster = xgb.Booster()
# booster.load_model(ART_DIR / "reg_log_totexpy2_xgb_es.booster.json")
# meta = json.loads((ART_DIR / "reg_log_totexpy2_xgb_es.meta.json").read_text())
# best_iter = meta["best_iteration"]

# Xp = pre.transform(X_new)
# d = xgb.DMatrix(Xp)
# yhat = booster.predict(d, iteration_range=(0, best_iter + 1))


# Feature contribution analysis on saved models

Recommended “main” method: permutation importance (model-agnostic)

This works across RF / XGBClassifier / MLP and is easy to compare.

In [144]:
import pandas as pd
from sklearn.inspection import permutation_importance

def perm_importance(pipeline, X, y, scoring, n_repeats=10, seed=42):
    r = permutation_importance(
        pipeline, X, y,
        n_repeats=n_repeats,
        random_state=seed,
        scoring=scoring
    )
    return pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False)


Suggested scoring:

Classification: scoring="average_precision" (aligns with PR-AUC)

Regression: scoring="neg_root_mean_squared_error" or "r2"

## 1. high cost

In [145]:
# HIGHCOST feature importance (RF pipeline)
imp_hc = perm_importance(rf_hc.model, X_test, y_test, scoring="average_precision")
imp_hc.head(15)


LOG_TOTEXPY1        0.116173
AGE                 0.044706
ANY_ED_Y1           0.030649
HIBPDXY1_BIN        0.030027
FAMSIZE_Y1_GRP      0.022936
FAMSIZE_Y1          0.021457
RTHLTH1_FAIRPOOR    0.020136
LOG_FAMINCY1        0.016586
POVCATY1_CAT        0.016114
REGIONY1_CAT        0.015369
CHOLDXY1_BIN        0.013304
RACE_ETH            0.012458
INS_TYPE_Y1         0.010945
ANY_IP_Y1           0.009377
WORKED_Y1           0.008986
dtype: float64

一个通用的分组分析函数（真实发生率 + 预测风险）

In [183]:
import numpy as np
import pandas as pd

def group_effect_tables(df, target, group_col, model_result=None):
    # ---------- actual (whole df) ----------
    tmp = df[[target, group_col]].dropna()
    overall_actual = tmp[target].mean()

    actual = (tmp.groupby(group_col, observed=True)[target]
              .agg(n="size", actual_rate="mean")
              .sort_values("actual_rate", ascending=False))
    actual["actual_rate_pct"] = (actual["actual_rate"] * 100).round(1)
    actual["lift_actual_vs_overall"] = (actual["actual_rate"] / overall_actual).round(2)
    actual["direction_actual_vs_overall"] = np.where(actual["actual_rate"] > overall_actual, "higher", "lower")

    pred = None
    if model_result is not None:
        pipe = model_result.model
        fit_cols = list(pipe.feature_names_in_)  # exact training columns + order

        # ✅ DO NOT duplicate group_col if it is already a model feature
        split_cols = fit_cols if group_col in fit_cols else fit_cols + [group_col]

        # dataset for splitting + grouping
        df2 = df[split_cols + [target]].dropna()

        X_all = df2[split_cols].copy()
        y_all = df2[target].astype(int)

        X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
            X_all, y_all, random_state=RANDOM_SEED, stratify=True
        )

        # predict using exact training columns
        proba = pipe.predict_proba(X_test[fit_cols])[:, 1]

        out = X_test[[group_col]].copy()
        out["proba"] = proba
        out["y_true"] = y_test.values

        overall_pred = out["proba"].mean()
        pred = (out.groupby(group_col, observed=True)
                .agg(n=("y_true", "size"),
                     pred_risk=("proba", "mean"),
                     actual_rate=("y_true", "mean"))
                .sort_values("pred_risk", ascending=False))

        pred["pred_risk_pct"] = (pred["pred_risk"] * 100).round(1)
        pred["actual_rate_pct"] = (pred["actual_rate"] * 100).round(1)
        pred["lift_pred_vs_overall"] = (pred["pred_risk"] / overall_pred).round(2)
        pred["direction_pred_vs_overall"] = np.where(pred["pred_risk"] > overall_pred, "higher", "lower")

    return actual, pred


In [184]:
rf_hc.model.feature_names_in_


array(['AGE', 'SEX_BIN', 'LOG_FAMINCY1', 'FAMSIZE_Y1', 'WORKED_Y1',
       'ANY_UNEMP_COMP_Y1', 'LOG_UNEMP_COMP_Y1', 'EMP_INFO_R12',
       'EMP_ATTACHED_ANY_R12_FILL0', 'RTHLTH1_FAIRPOOR',
       'MNHLTH1_FAIRPOOR', 'HIBPDXY1_BIN', 'CHDDXY1_BIN', 'STRKDXY1_BIN',
       'CHOLDXY1_BIN', 'ASTHDXY1_BIN', 'DIABDXY1_M18_BIN', 'LOG_TOTEXPY1',
       'ANY_ED_Y1', 'ANY_IP_Y1', 'RACE_ETH', 'REGIONY1_CAT', 'EDU_GROUP',
       'POVCATY1_CAT', 'FAMSIZE_Y1_GRP', 'INS_TYPE_Y1'], dtype=object)

看每个 age group 的 真实 HIGHCOST 发生率（最直观） 依旧 每个 age group 的 模型预测风险（更贴近“模型认为的影响”）

In [185]:
import pandas as pd

df_tmp = df_feat.copy()

df_tmp["AGE_GROUP"] = pd.cut(
    df_tmp["AGE"],
    bins=[0, 18, 30, 45, 65, 200],
    right=False,
    labels=["0-17", "18-29", "30-44", "45-64", "65+"]
)


In [186]:
act_age, pred_age = group_effect_tables(df_tmp, "HIGHCOST_Y2", "AGE_GROUP",
                                        model_result=rf_hc)
act_age, pred_age


(              n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 AGE_GROUP                                                               
 65+        1806     0.209302             20.9                    2.09   
 45-64      2123     0.121997             12.2                    1.22   
 30-44      1499     0.058706              5.9                    0.59   
 18-29       925     0.030270              3.0                    0.30   
 0-17       1459     0.019877              2.0                    0.20   
 
           direction_actual_vs_overall  
 AGE_GROUP                              
 65+                            higher  
 45-64                          higher  
 30-44                           lower  
 18-29                           lower  
 0-17                            lower  ,
              n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 AGE_GROUP                                                                
 65+        380   0.441916     0.192105   

FAMSIZE_Y1_GRP

In [198]:
act_ins, pred_ins = group_effect_tables(
    df_feat, "HIGHCOST_Y2", "FAMSIZE_Y1_GRP",
    model_result=rf_hc
)
act_ins, pred_ins

(                   n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 FAMSIZE_Y1_GRP                                                               
 1               1441     0.169327             16.9                    1.69   
 2               2246     0.139359             13.9                    1.39   
 3               1303     0.072909              7.3                    0.73   
 4               1445     0.053979              5.4                    0.54   
 5                785     0.044586              4.5                    0.45   
 6                363     0.030303              3.0                    0.30   
 7+               228     0.026316              2.6                    0.26   
 
                direction_actual_vs_overall  
 FAMSIZE_Y1_GRP                              
 1                                   higher  
 2                                   higher  
 3                                    lower  
 4                                    lower  
 5         

REGIONY1_CAT

In [199]:
act_ins, pred_ins = group_effect_tables(
    df_feat, "HIGHCOST_Y2", "REGIONY1_CAT",
    model_result=rf_hc
)
act_ins, pred_ins

(                 n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 REGIONY1_CAT                                                               
 Northeast     1203     0.134663             13.5                    1.35   
 South         3062     0.095363              9.5                    0.95   
 Midwest       1574     0.095299              9.5                    0.95   
 West          1973     0.090218              9.0                    0.90   
 
              direction_actual_vs_overall  
 REGIONY1_CAT                              
 Northeast                         higher  
 South                              lower  
 Midwest                            lower  
 West                               lower  ,
                 n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 REGIONY1_CAT                                                                
 Northeast     243   0.297771     0.144033           29.8             14.4   
 Midwest       298   0.272855     0.1

RACE_ETH

In [200]:
act_ins, pred_ins = group_effect_tables(
    df_feat, "HIGHCOST_Y2", "RACE_ETH",
    model_result=rf_hc
)
act_ins, pred_ins

(                      n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 RACE_ETH                                                                        
 NH White           4172     0.126798             12.7                    1.27   
 NH Black           1140     0.092982              9.3                    0.93   
 NH Other/multiple   267     0.067416              6.7                    0.67   
 NH Asian            475     0.063158              6.3                    0.63   
 Hispanic           1758     0.056314              5.6                    0.56   
 
                   direction_actual_vs_overall  
 RACE_ETH                                       
 NH White                               higher  
 NH Black                                lower  
 NH Other/multiple                       lower  
 NH Asian                                lower  
 Hispanic                                lower  ,
                      n  pred_risk  actual_rate  pred_risk_pct  \
 RACE_ETH     

To assess directional effects for categorical and grouped predictors, we complemented permutation importance with subgroup summaries of (i) observed HIGHCOST_Y2 prevalence and (ii) mean predicted risk from the selected RF model. High-cost prevalence increases markedly with age: individuals aged 65+ show an observed rate of 20.9% (lift 2.09 vs overall), while 45–64 also exceeds the overall average (12.2%, lift 1.22). Predicted risks follow the same ranking. Family size shows an inverse gradient: one- and two-person families have higher observed rates (16.9% and 13.9%) than larger households, and the model similarly assigns higher predicted risk to smaller families. Regional differences are smaller but consistent, with the Northeast exhibiting higher observed and predicted risk than other regions. Differences by race/ethnicity are also present; however, these patterns are interpreted as associations that may reflect underlying differences in socioeconomic status, health burden, and access to care rather than causal effects, and thus require cautious interpretation.

INS_TYPE_Y1（类别变量）

In [187]:
act_ins, pred_ins = group_effect_tables(
    df_feat, "HIGHCOST_Y2", "INS_TYPE_Y1",
    model_result=rf_hc
)
act_ins, pred_ins


(                                n  actual_rate  actual_rate_pct  \
 INS_TYPE_Y1                                                       
 65+ Medicare + other public   310     0.283871             28.4   
 65+ other coverage             29     0.241379             24.1   
 65+ Medicare + private        759     0.202899             20.3   
 65+ Medicare only             699     0.184549             18.5   
 <65 public only              1547     0.074984              7.5   
 <65 any private              3905     0.070679              7.1   
 <65 uninsured                 554     0.021661              2.2   
 65+ uninsured                   9     0.000000              0.0   
 
                              lift_actual_vs_overall  \
 INS_TYPE_Y1                                           
 65+ Medicare + other public                    2.84   
 65+ other coverage                             2.41   
 65+ Medicare + private                         2.03   
 65+ Medicare only                    

In [188]:
act_pov, pred_pov = group_effect_tables(df_feat, "HIGHCOST_Y2", "POVCATY1_CAT",
                                        model_result=rf_hc)
act_pov, pred_pov


(                    n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 POVCATY1_CAT                                                                  
 Near poor         372     0.139785             14.0                    1.40   
 Poor / negative  1191     0.115869             11.6                    1.16   
 High income      2977     0.098421              9.8                    0.98   
 Low income       1093     0.094236              9.4                    0.94   
 Middle income    2179     0.089950              9.0                    0.90   
 
                 direction_actual_vs_overall  
 POVCATY1_CAT                                 
 Near poor                            higher  
 Poor / negative                      higher  
 High income                           lower  
 Low income                            lower  
 Middle income                         lower  ,
                    n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 POVCATY1_CAT              

RTHLTH1_FAIRPOOR（二元：健康差/一般 vs 好/很好/极好）

In [189]:
act_hlth, pred_hlth = group_effect_tables(df_feat, "HIGHCOST_Y2", "RTHLTH1_FAIRPOOR",
                                         model_result=rf_hc)
act_hlth, pred_hlth


(                     n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 RTHLTH1_FAIRPOOR                                                               
 1                 1029     0.241011             24.1                    2.41   
 0                 6783     0.078726              7.9                    0.79   
 
                  direction_actual_vs_overall  
 RTHLTH1_FAIRPOOR                              
 1                                     higher  
 0                                      lower  ,
                      n  pred_risk  actual_rate  pred_risk_pct  \
 RTHLTH1_FAIRPOOR                                                
 1                  224   0.446756     0.250000           44.7   
 0                 1266   0.215855     0.077409           21.6   
 
                   actual_rate_pct  lift_pred_vs_overall  \
 RTHLTH1_FAIRPOOR                                          
 1                            25.0                  1.78   
 0                             7.7 

慢病（二元）：HIBPDXY1_BIN / CHOLDXY1_BIN

In [192]:
for col in ["HIBPDXY1_BIN", "CHOLDXY1_BIN"]:
    act_c, pred_c = group_effect_tables(df_feat, "HIGHCOST_Y2", col,
                                        model_result=rf_hc)
    print("\n====", col, "====")
    display(act_c,pred_c)



==== HIBPDXY1_BIN ====


,n,actual_rate,actual_rate_pct,lift_actual_vs_overall,direction_actual_vs_overall
HIBPDXY1_BIN,,,,,
1,2364,0.197970,19.8,1.98,higher
0,5448,0.057636,5.8,0.58,lower


,n,pred_risk,actual_rate,pred_risk_pct,actual_rate_pct,lift_pred_vs_overall,direction_pred_vs_overall
HIBPDXY1_BIN,,,,,,,
1,500,0.422918,0.202000,42.3,20.2,1.69,higher
0,990,0.163522,0.053535,16.4,5.4,0.65,lower



==== CHOLDXY1_BIN ====


,n,actual_rate,actual_rate_pct,lift_actual_vs_overall,direction_actual_vs_overall
CHOLDXY1_BIN,,,,,
1,2223,0.190283,19.0,1.90,higher
0,5589,0.064233,6.4,0.64,lower


,n,pred_risk,actual_rate,pred_risk_pct,actual_rate_pct,lift_pred_vs_overall,direction_pred_vs_overall
CHOLDXY1_BIN,,,,,,,
1,453,0.403169,0.189845,40.3,19.0,1.61,higher
0,1037,0.183906,0.065574,18.4,6.6,0.73,lower


High-cost risk increases sharply with age and is concentrated in Medicare-linked insurance categories; it is also higher among individuals reporting fair/poor baseline health and among those with chronic conditions such as hypertension and high cholesterol. Socioeconomic indicators (poverty category) show weaker gradients in model-predicted risk, suggesting that their predictive contribution is partly mediated through correlated clinical and utilization variables. These results are interpreted as associations rather than causal effects.

基线利用（二元）：ANY_ED_Y1 / ANY_IP_Y1

In [193]:
for col in ["ANY_ED_Y1", "ANY_IP_Y1"]:
    act_c, pred_c = group_effect_tables(df_feat, "HIGHCOST_Y2", col,
                                        model_result=rf_hc)
    print("\n====", col, "====")
    display(act_c,pred_c)



==== ANY_ED_Y1 ====


,n,actual_rate,actual_rate_pct,lift_actual_vs_overall,direction_actual_vs_overall
ANY_ED_Y1,,,,,
1,1093,0.220494,22.0,2.2,higher
0,6719,0.080518,8.1,0.8,lower


,n,pred_risk,actual_rate,pred_risk_pct,actual_rate_pct,lift_pred_vs_overall,direction_pred_vs_overall
ANY_ED_Y1,,,,,,,
1,205,0.449254,0.224390,44.9,22.4,1.79,higher
0,1285,0.218871,0.084047,21.9,8.4,0.87,lower



==== ANY_IP_Y1 ====


,n,actual_rate,actual_rate_pct,lift_actual_vs_overall,direction_actual_vs_overall
ANY_IP_Y1,,,,,
1,464,0.314655,31.5,3.14,higher
0,7348,0.086554,8.7,0.86,lower


,n,pred_risk,actual_rate,pred_risk_pct,actual_rate_pct,lift_pred_vs_overall,direction_pred_vs_overall
ANY_IP_Y1,,,,,,,
1,92,0.614672,0.336957,61.5,33.7,2.45,higher
0,1398,0.226607,0.087983,22.7,8.8,0.90,lower


LOG_TOTEXPY1（按四分位）

In [194]:
df_tmp = df_feat.copy()
df_tmp["LOG_TOTEXPY1_Q"] = pd.qcut(df_tmp["LOG_TOTEXPY1"], q=4, labels=["Q1(low)","Q2","Q3","Q4(high)"])

act_q, pred_q = group_effect_tables(df_tmp, "HIGHCOST_Y2", "LOG_TOTEXPY1_Q",
                                    model_result=rf_hc)
act_q, pred_q


(                   n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 LOG_TOTEXPY1_Q                                                               
 Q4(high)        1953     0.294419             29.4                    2.94   
 Q3              1953     0.059396              5.9                    0.59   
 Q2              1953     0.030722              3.1                    0.31   
 Q1(low)         1953     0.015873              1.6                    0.16   
 
                direction_actual_vs_overall  
 LOG_TOTEXPY1_Q                              
 Q4(high)                            higher  
 Q3                                   lower  
 Q2                                   lower  
 Q1(low)                              lower  ,
                   n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 LOG_TOTEXPY1_Q                                                                
 Q4(high)        391   0.539494     0.286445           53.9             28.6   
 Q3    

LOG_FAMINCY1（按四分位）

In [195]:
df_tmp = df_feat.copy()
df_tmp["LOG_FAMINCY1_Q"] = pd.qcut(df_tmp["LOG_FAMINCY1"], q=4, labels=["Q1(low)","Q2","Q3","Q4(high)"])

act_inc, pred_inc = group_effect_tables(df_tmp, "HIGHCOST_Y2", "LOG_FAMINCY1_Q",
                                        model_result=rf_hc)
act_inc, pred_inc


(                   n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 LOG_FAMINCY1_Q                                                               
 Q1(low)         1953     0.140809             14.1                    1.41   
 Q3              1954     0.093142              9.3                    0.93   
 Q2              1953     0.084485              8.4                    0.84   
 Q4(high)        1952     0.081967              8.2                    0.82   
 
                direction_actual_vs_overall  
 LOG_FAMINCY1_Q                              
 Q1(low)                             higher  
 Q3                                   lower  
 Q2                                   lower  
 Q4(high)                             lower  ,
                   n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 LOG_FAMINCY1_Q                                                                
 Q1(low)         367   0.296755     0.117166           29.7             11.7   
 Q3    

Taken together, these subgroup analyses provide directional interpretation complementary to permutation importance. The largest and most consistent risk gradients are driven by prior-year utilization and prior-year expenditures, while socio-economic variables (e.g., income, poverty category) show smaller marginal differences in model-predicted risk, suggesting shared variance with clinical and utilization features.

## 2. ANY_ED_Y2

In [201]:
imp_ed = perm_importance(xgb_ed.model, X_test, y_test, scoring="average_precision")
imp_ed.head(10)

LOG_TOTEXPY1         0.083875
ANY_ED_Y1            0.052691
LOG_FAMINCY1         0.039297
AGE                  0.037340
LOG_UNEMP_COMP_Y1    0.017618
REGIONY1_CAT         0.016025
SEX_BIN              0.010115
POVCATY1_CAT         0.008676
INS_TYPE_Y1          0.007457
WORKED_Y1            0.006629
dtype: float64

In [202]:
df_tmp = df_feat.copy()

df_tmp["AGE_GROUP"] = pd.cut(
    df_tmp["AGE"],
    bins=[0, 18, 30, 45, 65, 200],
    right=False,
    labels=["0-17","18-29","30-44","45-64","65+"]
)

df_tmp["LOG_TOTEXPY1_Q"] = pd.qcut(df_tmp["LOG_TOTEXPY1"], q=4, labels=["Q1(low)","Q2","Q3","Q4(high)"])
df_tmp["LOG_FAMINCY1_Q"] = pd.qcut(df_tmp["LOG_FAMINCY1"], q=4, labels=["Q1(low)","Q2","Q3","Q4(high)"])


In [203]:
# 1) baseline ED use (最重要的方向性)
act, pred = group_effect_tables(df_tmp, "ANY_ED_Y2", "ANY_ED_Y1", model_result=xgb_ed)
act, pred


(              n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 ANY_ED_Y1                                                               
 1          1093     0.335773             33.6                    2.35   
 0          6719     0.111326             11.1                    0.78   
 
           direction_actual_vs_overall  
 ANY_ED_Y1                              
 1                              higher  
 0                               lower  ,
               n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 ANY_ED_Y1                                                                 
 1           217   0.323507     0.304147      32.400002             30.4   
 0          1273   0.111225     0.115475      11.100000             11.5   
 
            lift_pred_vs_overall direction_pred_vs_overall  
 ANY_ED_Y1                                                  
 1                          2.28                    higher  
 0                          0.78                

In [204]:
# 2) age group
act_age, pred_age = group_effect_tables(df_tmp, "ANY_ED_Y2", "AGE_GROUP", model_result=xgb_ed)

# 3) baseline total expenditure quartiles
act_tx, pred_tx = group_effect_tables(df_tmp, "ANY_ED_Y2", "LOG_TOTEXPY1_Q", model_result=xgb_ed)

# 4) income quartiles (或 poverty)
act_inc, pred_inc = group_effect_tables(df_tmp, "ANY_ED_Y2", "LOG_FAMINCY1_Q", model_result=xgb_ed)
# 或者：
act_pov, pred_pov = group_effect_tables(df_tmp, "ANY_ED_Y2", "POVCATY1_CAT", model_result=xgb_ed)

# 5) unemployment compensation (binary)
act_un, pred_un = group_effect_tables(df_tmp, "ANY_ED_Y2", "ANY_UNEMP_COMP_Y1", model_result=xgb_ed)

# 6) insurance type & region
act_ins, pred_ins = group_effect_tables(df_tmp, "ANY_ED_Y2", "INS_TYPE_Y1", model_result=xgb_ed)
act_reg, pred_reg = group_effect_tables(df_tmp, "ANY_ED_Y2", "REGIONY1_CAT", model_result=xgb_ed)


In [205]:
act_age, pred_age

(              n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 AGE_GROUP                                                               
 65+        1806     0.215947             21.6                    1.51   
 45-64      2123     0.151201             15.1                    1.06   
 0-17       1459     0.110350             11.0                    0.77   
 30-44      1499     0.101401             10.1                    0.71   
 18-29       925     0.098378              9.8                    0.69   
 
           direction_actual_vs_overall  
 AGE_GROUP                              
 65+                            higher  
 45-64                          higher  
 0-17                            lower  
 30-44                           lower  
 18-29                           lower  ,
              n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 AGE_GROUP                                                                
 65+        365   0.218682     0.213699   

In [206]:
act_tx, pred_tx

(                   n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 LOG_TOTEXPY1_Q                                                               
 Q4(high)        1953     0.255504             25.6                    1.79   
 Q3              1953     0.140297             14.0                    0.98   
 Q2              1953     0.111111             11.1                    0.78   
 Q1(low)         1953     0.064004              6.4                    0.45   
 
                direction_actual_vs_overall  
 LOG_TOTEXPY1_Q                              
 Q4(high)                            higher  
 Q3                                   lower  
 Q2                                   lower  
 Q1(low)                              lower  ,
                   n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 LOG_TOTEXPY1_Q                                                                
 Q4(high)        393   0.246971     0.259542      24.700001             26.0   
 Q3    

In [207]:
act_inc, pred_inc

(                   n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 LOG_FAMINCY1_Q                                                               
 Q1(low)         1953     0.206861             20.7                    1.45   
 Q2              1953     0.141321             14.1                    0.99   
 Q3              1954     0.119243             11.9                    0.84   
 Q4(high)        1952     0.103484             10.3                    0.73   
 
                direction_actual_vs_overall  
 LOG_FAMINCY1_Q                              
 Q1(low)                             higher  
 Q2                                   lower  
 Q3                                   lower  
 Q4(high)                             lower  ,
                   n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 LOG_FAMINCY1_Q                                                                
 Q1(low)         388   0.200802     0.201031           20.1             20.1   
 Q2    

In [208]:
act_pov, pred_pov

(                    n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 POVCATY1_CAT                                                                  
 Near poor         372     0.201613             20.2                    1.41   
 Poor / negative  1191     0.190596             19.1                    1.34   
 Low income       1093     0.160110             16.0                    1.12   
 Middle income    2179     0.135383             13.5                    0.95   
 High income      2977     0.115217             11.5                    0.81   
 
                 direction_actual_vs_overall  
 POVCATY1_CAT                                 
 Near poor                            higher  
 Poor / negative                      higher  
 Low income                           higher  
 Middle income                         lower  
 High income                           lower  ,
                    n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 POVCATY1_CAT              

In [209]:
act_un, pred_un

(                      n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 ANY_UNEMP_COMP_Y1                                                               
 1                   106     0.245283             24.5                    1.72   
 0                  7706     0.141318             14.1                    0.99   
 
                   direction_actual_vs_overall  
 ANY_UNEMP_COMP_Y1                              
 1                                      higher  
 0                                       lower  ,
                       n  pred_risk  actual_rate  pred_risk_pct  \
 ANY_UNEMP_COMP_Y1                                                
 1                    24   0.329579     0.375000           33.0   
 0                  1466   0.139073     0.139154           13.9   
 
                    actual_rate_pct  lift_pred_vs_overall  \
 ANY_UNEMP_COMP_Y1                                          
 1                             37.5                  2.32   
 0                  

In [210]:
act_ins, pred_ins 

(                                n  actual_rate  actual_rate_pct  \
 INS_TYPE_Y1                                                       
 65+ Medicare + other public   310     0.274194             27.4   
 65+ Medicare only             699     0.231760             23.2   
 65+ Medicare + private        759     0.185771             18.6   
 <65 public only              1547     0.160310             16.0   
 <65 any private              3905     0.107298             10.7   
 <65 uninsured                 554     0.104693             10.5   
 65+ other coverage             29     0.068966              6.9   
 65+ uninsured                   9     0.000000              0.0   
 
                              lift_actual_vs_overall  \
 INS_TYPE_Y1                                           
 65+ Medicare + other public                    1.92   
 65+ Medicare only                              1.62   
 65+ Medicare + private                         1.30   
 <65 public only                      

In [211]:
act_reg, pred_reg

(                 n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 REGIONY1_CAT                                                               
 Midwest       1574     0.153113             15.3                    1.07   
 South         3062     0.150229             15.0                    1.05   
 Northeast     1203     0.135495             13.5                    0.95   
 West          1973     0.127217             12.7                    0.89   
 
              direction_actual_vs_overall  
 REGIONY1_CAT                              
 Midwest                           higher  
 South                             higher  
 Northeast                          lower  
 West                               lower  ,
                 n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 REGIONY1_CAT                                                                
 Midwest       296   0.154244     0.152027           15.4             15.2   
 South         606   0.150694     0.1

## 3. ANY_IP_Y2

In [212]:
imp_ip = perm_importance(rf_ip_tuned.model, X_test, y_test, scoring="average_precision")
imp_ip.head(10)

LOG_TOTEXPY1        0.045603
ANY_ED_Y1           0.028448
HIBPDXY1_BIN        0.014620
AGE                 0.014307
SEX_BIN             0.007186
RTHLTH1_FAIRPOOR    0.003961
DIABDXY1_M18_BIN    0.003371
FAMSIZE_Y1_GRP      0.003286
STRKDXY1_BIN        0.001762
LOG_FAMINCY1        0.001431
dtype: float64

In [213]:
df_tmp = df_feat.copy()

# Age groups
df_tmp["AGE_GROUP"] = pd.cut(
    df_tmp["AGE"],
    bins=[0, 18, 30, 45, 65, 200],
    right=False,
    labels=["0-17", "18-29", "30-44", "45-64", "65+"]
)

# Prior-year cost quartiles
df_tmp["LOG_TOTEXPY1_Q"] = pd.qcut(
    df_tmp["LOG_TOTEXPY1"],
    q=4,
    labels=["Q1(low)", "Q2", "Q3", "Q4(high)"]
)


In [214]:
# (1) baseline cost quartile
act_cost, pred_cost = group_effect_tables(
    df_tmp, "ANY_IP_Y2", "LOG_TOTEXPY1_Q",
    model_result=rf_ip_tuned
)

# (2) age group
act_age, pred_age = group_effect_tables(
    df_tmp, "ANY_IP_Y2", "AGE_GROUP",
    model_result=rf_ip_tuned
)

# (3) hypertension (binary)
act_htn, pred_htn = group_effect_tables(
    df_tmp, "ANY_IP_Y2", "HIBPDXY1_BIN",
    model_result=rf_ip_tuned
)

 


In [215]:
act_cost, pred_cost

(                   n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 LOG_TOTEXPY1_Q                                                               
 Q4(high)        1953     0.160266             16.0                    2.21   
 Q3              1953     0.059396              5.9                    0.82   
 Q2              1953     0.047619              4.8                    0.66   
 Q1(low)         1953     0.023041              2.3                    0.32   
 
                direction_actual_vs_overall  
 LOG_TOTEXPY1_Q                              
 Q4(high)                            higher  
 Q3                                   lower  
 Q2                                   lower  
 Q1(low)                              lower  ,
                   n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 LOG_TOTEXPY1_Q                                                                
 Q4(high)        389   0.361602     0.167095           36.2             16.7   
 Q3    

In [216]:
act_age, pred_age

(              n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 AGE_GROUP                                                               
 65+        1806     0.156700             15.7                    2.16   
 45-64      2123     0.071597              7.2                    0.99   
 30-44      1499     0.045364              4.5                    0.63   
 18-29       925     0.041081              4.1                    0.57   
 0-17       1459     0.017820              1.8                    0.25   
 
           direction_actual_vs_overall  
 AGE_GROUP                              
 65+                            higher  
 45-64                           lower  
 30-44                           lower  
 18-29                           lower  
 0-17                            lower  ,
              n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 AGE_GROUP                                                                
 65+        343   0.366887     0.177843   

In [217]:
act_htn, pred_htn

(                 n  actual_rate  actual_rate_pct  lift_actual_vs_overall  \
 HIBPDXY1_BIN                                                               
 1             2364     0.142978             14.3                    1.97   
 0             5448     0.042034              4.2                    0.58   
 
              direction_actual_vs_overall  
 HIBPDXY1_BIN                              
 1                                 higher  
 0                                  lower  ,
                  n  pred_risk  actual_rate  pred_risk_pct  actual_rate_pct  \
 HIBPDXY1_BIN                                                                 
 1              454   0.319905     0.134361           32.0             13.4   
 0             1036   0.135922     0.049228           13.6              4.9   
 
               lift_pred_vs_overall direction_pred_vs_overall  
 HIBPDXY1_BIN                                                  
 1                             1.67                    higher  
 0 

## 4. LOG_TOTEXPY2

In [219]:
import xgboost as xgb
import joblib, json
from pathlib import Path
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

ART_DIR = Path("results/model_artifacts")

# load preprocess + booster + meta
pre = joblib.load(ART_DIR / "reg_log_totexpy2_xgb_es.preprocess.joblib")

booster = xgb.Booster()
booster.load_model(ART_DIR / "reg_log_totexpy2_xgb_es.booster.json")

meta = json.loads((ART_DIR / "reg_log_totexpy2_xgb_es.meta.json").read_text())
best_iter = int(meta["best_iteration"])

# rebuild X,y exactly for regression
X = df_feat[num_cols + cat_cols].copy()
y = df_feat["LOG_TOTEXPY2"].copy()
mask = y.notna()
X, y = X.loc[mask], y.loc[mask]

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=RANDOM_SEED, stratify=False
)

# predict on test using best iteration range
Xp = pre.transform(X_test)
dtest = xgb.DMatrix(Xp)
yhat = booster.predict(dtest, iteration_range=(0, best_iter + 1))

rmse = root_mean_squared_error(y_test, yhat)
mae = mean_absolute_error(y_test, yhat)
r2 = r2_score(y_test, yhat)

print("Loaded model metrics:", {"RMSE_log": rmse, "MAE_log": mae, "R2": r2})


Loaded model metrics: {'RMSE_log': 2.1500502264902157, 'MAE_log': 1.5415488342366697, 'R2': 0.5297353608086273}


In [220]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error

def perm_importance_xgb_booster_rmse(pre, booster, best_iter, X_ref, y_ref, n_repeats=5, seed=42):
    rng = np.random.default_rng(seed)

    # baseline
    d0 = xgb.DMatrix(pre.transform(X_ref))
    yhat0 = booster.predict(d0, iteration_range=(0, best_iter + 1))
    base_rmse = root_mean_squared_error(y_ref, yhat0)

    out = {}
    for col in X_ref.columns:
        rmses = []
        for _ in range(n_repeats):
            Xp = X_ref.copy()
            Xp[col] = rng.permutation(Xp[col].values)

            d = xgb.DMatrix(pre.transform(Xp))
            yhat = booster.predict(d, iteration_range=(0, best_iter + 1))
            rmses.append(root_mean_squared_error(y_ref, yhat))

        out[col] = float(np.mean(rmses) - base_rmse)  # positive = important

    imp = pd.Series(out).sort_values(ascending=False)
    return base_rmse, imp

base_rmse, imp = perm_importance_xgb_booster_rmse(
    pre, booster, best_iter,
    X_test, y_test,
    n_repeats=5,
    seed=RANDOM_SEED
)

print("baseline RMSE_log:", base_rmse)
imp.head(10)


baseline RMSE_log: 2.1500502264902157


LOG_TOTEXPY1    0.996374
AGE             0.056103
RACE_ETH        0.049515
INS_TYPE_Y1     0.032864
FAMSIZE_Y1      0.021291
ANY_IP_Y1       0.019959
SEX_BIN         0.018577
LOG_FAMINCY1    0.013214
EDU_GROUP       0.011110
CHOLDXY1_BIN    0.009550
dtype: float64

In [221]:
import pandas as pd
import numpy as np

df_tmp = df_feat.copy()

# age bands
df_tmp["AGE_GROUP"] = pd.cut(
    df_tmp["AGE"],
    bins=[0, 18, 30, 45, 65, 200],
    right=False,
    labels=["0-17","18-29","30-44","45-64","65+"]
)

# prior-year expenditure quartiles
df_tmp["LOG_TOTEXPY1_Q"] = pd.qcut(
    df_tmp["LOG_TOTEXPY1"],
    q=4,
    labels=["Q1(low)","Q2","Q3","Q4(high)"]
)




In [222]:
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error

def subgroup_effect_regression(
    df, y_col, group_col,
    pre, booster, best_iter,
    feature_cols,
    *,
    random_state=42
):
    """
    Returns:
      - actual table: group sizes + mean(y) + delta vs overall + direction
      - predicted table (on test split): mean(pred) + mean(actual) + deltas + direction
    """

    # ---------- actual (whole df) ----------
    tmp = df[[y_col, group_col]].dropna()
    overall_y = tmp[y_col].mean()

    actual = (tmp.groupby(group_col, observed=True)[y_col]
              .agg(n="size", mean_actual="mean")
              .sort_values("mean_actual", ascending=False))
    actual["delta_actual_vs_overall"] = (actual["mean_actual"] - overall_y)
    actual["direction_actual_vs_overall"] = np.where(actual["mean_actual"] > overall_y, "higher", "lower")

    # ---------- predicted (test split only, to match your model evaluation logic) ----------
    X = df[feature_cols].copy()
    y = df[y_col].copy()
    mask = y.notna()
    X, y = X.loc[mask], y.loc[mask]

    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        X, y, random_state=random_state, stratify=False
    )

    # need group labels aligned to X_test index
    g_test = df.loc[X_test.index, group_col]

    Xp_test = pre.transform(X_test)
    dtest = xgb.DMatrix(Xp_test)
    yhat = booster.predict(dtest, iteration_range=(0, int(best_iter) + 1))

    out = pd.DataFrame({
        group_col: g_test.values,
        "y_true": y_test.values,
        "y_pred": yhat
    }).dropna(subset=[group_col])

    overall_pred = out["y_pred"].mean()
    overall_true = out["y_true"].mean()

    pred = (out.groupby(group_col, observed=True)
            .agg(n=("y_true","size"),
                 mean_pred=("y_pred","mean"),
                 mean_actual=("y_true","mean"))
            .sort_values("mean_pred", ascending=False))

    pred["delta_pred_vs_overall"] = pred["mean_pred"] - overall_pred
    pred["delta_actual_vs_overall"] = pred["mean_actual"] - overall_true
    pred["direction_pred_vs_overall"] = np.where(pred["mean_pred"] > overall_pred, "higher", "lower")

    return actual, pred


In [223]:
# feature cols used by regression model
feature_cols = num_cols + cat_cols  # 你回归训练用的就是这一组

# 1) baseline cost quartiles
act_cost, pred_cost = subgroup_effect_regression(
    df_tmp, "LOG_TOTEXPY2", "LOG_TOTEXPY1_Q",
    pre, booster, best_iter,
    feature_cols,
    random_state=RANDOM_SEED
)

# 2) age bands
act_age, pred_age = subgroup_effect_regression(
    df_tmp, "LOG_TOTEXPY2", "AGE_GROUP",
    pre, booster, best_iter,
    feature_cols,
    random_state=RANDOM_SEED
)






In [224]:
act_cost, pred_cost

(                   n  mean_actual  delta_actual_vs_overall  \
 LOG_TOTEXPY1_Q                                               
 Q4(high)        1953     9.024703                 2.355679   
 Q3              1953     7.652090                 0.983067   
 Q2              1953     6.432131                -0.236893   
 Q1(low)         1953     3.567170                -3.101854   
 
                direction_actual_vs_overall  
 LOG_TOTEXPY1_Q                              
 Q4(high)                            higher  
 Q3                                  higher  
 Q2                                   lower  
 Q1(low)                              lower  ,
                   n  mean_pred  mean_actual  delta_pred_vs_overall  \
 LOG_TOTEXPY1_Q                                                       
 Q4(high)        412   9.026003     8.952715               2.272295   
 Q3              393   7.606519     7.701391               0.852810   
 Q2              397   6.420806     6.465658              -

In [225]:
act_age, pred_age

(              n  mean_actual  delta_actual_vs_overall  \
 AGE_GROUP                                               
 65+        1806     8.476856                 1.807833   
 45-64      2123     7.185135                 0.516111   
 30-44      1499     5.791888                -0.877135   
 0-17       1459     5.640378                -1.028645   
 18-29       925     4.998719                -1.670304   
 
           direction_actual_vs_overall  
 AGE_GROUP                              
 65+                            higher  
 45-64                          higher  
 30-44                           lower  
 0-17                            lower  
 18-29                           lower  ,
              n  mean_pred  mean_actual  delta_pred_vs_overall  \
 AGE_GROUP                                                       
 65+        377   8.473631     8.427697               1.719923   
 45-64      423   7.273368     7.240052               0.519660   
 30-44      308   5.946897     5.895549 

## Error analysis

In [261]:
PROJECT_ROOT = Path("..") 

In [262]:
ART_DIR = PROJECT_ROOT / "results" / "model_artifacts"

In [263]:
# HIGHCOST RF
clf_highcost = joblib.load(ART_DIR / "clf_highcost_rf.joblib")
meta_highcost = json.loads((ART_DIR / "clf_highcost_rf.meta.json").read_text())
t_highcost = meta_highcost["best_threshold"]

# ED XGB
clf_ed = joblib.load(ART_DIR / "clf_ed_xgb.joblib")
meta_ed = json.loads((ART_DIR / "clf_ed_xgb.meta.json").read_text())
t_ed = meta_ed["best_threshold"]

# IP RF
clf_ip = joblib.load(ART_DIR / "clf_ip_rf.joblib")
meta_ip = json.loads((ART_DIR / "clf_ip_rf.meta.json").read_text())
t_ip = meta_ip["best_threshold"]

t_highcost, t_ed, t_ip


(0.5499999999999999, 0.2, 0.39999999999999997)

In [264]:
pre_reg = joblib.load(ART_DIR / "reg_log_totexpy2_xgb_es.preprocess.joblib")

booster = xgb.Booster()
booster.load_model(ART_DIR / "reg_log_totexpy2_xgb_es.booster.json")

meta_reg = json.loads((ART_DIR / "reg_log_totexpy2_xgb_es.meta.json").read_text())
best_iter = int(meta_reg["best_iteration"])

best_iter

1037

In [269]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from src.models import split_train_val_test

In [277]:
def classification_error_table(
    df_feat: pd.DataFrame,
    target: str,
    feature_cols: list[str],
    fitted_model,                 # sklearn Pipeline with predict_proba
    threshold: float,
    *,
    random_state: int = 42,
    stratify: bool = True
):
    # ✅ only filter on target, keep feature missingness (imputer handles it)
    y_raw = df_feat[target]
    mask = y_raw.notna()

    X = df_feat.loc[mask, feature_cols].copy()
    y = y_raw.loc[mask].astype(int)

    X_tr, X_va, X_te, y_tr, y_va, y_te = split_train_val_test(
        X, y, random_state=random_state, stratify=stratify
    )

    proba = fitted_model.predict_proba(X_te)[:, 1]
    y_pred = (proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    cm_dict = {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

    df_err = X_te.copy()
    df_err["y_true"] = y_te.values
    df_err["proba"] = proba
    df_err["y_pred"] = y_pred

    def _etype(r):
        if r.y_true == 1 and r.y_pred == 1: return "TP"
        if r.y_true == 0 and r.y_pred == 1: return "FP"
        if r.y_true == 1 and r.y_pred == 0: return "FN"
        return "TN"

    df_err["error_type"] = df_err.apply(_etype, axis=1)
    return df_err, cm_dict


In [278]:
# ===== Example: ANY_IP_Y2 (recommended for error analysis because hardest) =====
TARGET = "ANY_IP_Y2"
FEATURE_COLS = FEATURES          # 你论文最终特征：num_cols + cat_cols
THRESHOLD = t_ip            # 用你主文的 operating point

df_ip_err, cm_ip = classification_error_table(
    df_feat, TARGET, FEATURE_COLS,
    fitted_model=clf_ip,         
    threshold=THRESHOLD,
    random_state=42,
    stratify=True
)

cm_ip, df_ip_err["error_type"].value_counts()

({'TN': 1316, 'FP': 134, 'FN': 72, 'TP': 41},
 error_type
 TN    1316
 FP     134
 FN      72
 TP      41
 Name: count, dtype: int64)

In [270]:
# df_ip_err 已经有 proba 和 error_type
df_ip_err.groupby("error_type")["proba"].describe()[["count","mean","std","min","25%","50%","75%","max"]]


,count,mean,std,min,25%,50%,75%,max
error_type,,,,,,,,
FN,72.0,0.210492,0.099938,0.011339,0.133720,0.212458,0.294413,0.392389
FP,134.0,0.514135,0.097376,0.400996,0.441352,0.481654,0.554262,0.814296
TN,1316.0,0.140883,0.099374,0.003552,0.059556,0.115575,0.202919,0.396389
TP,41.0,0.565472,0.112943,0.403278,0.479866,0.561104,0.665178,0.824710


In [273]:
fp = df_ip_err[df_ip_err["error_type"]=="FP"]
fn = df_ip_err[df_ip_err["error_type"]=="FN"]

# 1) numeric mean comparison
num_cols_small = ["LOG_TOTEXPY1", "AGE"]
num_cmp = pd.DataFrame({
    "FP_mean": fp[num_cols_small].mean(numeric_only=True),
    "FN_mean": fn[num_cols_small].mean(numeric_only=True),
})
num_cmp["diff_FP_minus_FN"] = num_cmp["FP_mean"] - num_cmp["FN_mean"]
num_cmp


,FP_mean,FN_mean,diff_FP_minus_FN
LOG_TOTEXPY1,9.749797,7.603151,2.146646
AGE,68.298507,49.527778,18.770730


In [274]:
# 2) binary rate comparison (mean of 0/1)
bin_cols_small = ["ANY_ED_Y1", "HIBPDXY1_BIN"]
bin_cmp = pd.DataFrame({
    "FP_rate": fp[bin_cols_small].mean(numeric_only=True),
    "FN_rate": fn[bin_cols_small].mean(numeric_only=True),
})
bin_cmp["diff_FP_minus_FN"] = bin_cmp["FP_rate"] - bin_cmp["FN_rate"]
bin_cmp


,FP_rate,FN_rate,diff_FP_minus_FN
ANY_ED_Y1,0.402985,0.194444,0.208541
HIBPDXY1_BIN,0.753731,0.416667,0.337065


## fairness 

In [287]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

def group_fairness_audit(
    df_feat: pd.DataFrame,
    target: str,
    feature_cols: list[str],
    model,                     # sklearn Pipeline with predict_proba
    group_col: str,
    *,
    random_state: int = 42,
    top_fracs=(0.05, 0.10, 0.20),
    stratify: bool = True,
):
    """
    Minimal fairness audit:
      - group-wise PR-AUC (average precision within group, if both classes exist)
      - recall@top-k under a GLOBAL top-k policy (flag top k% overall, then measure recall within each group)

    Notes:
      - We only drop missing target/group values; feature missingness is kept for the pipeline imputers.
      - Handles the case where group_col is already included in feature_cols (e.g., SEX_BIN).
    """

    # ---- 0) de-duplicate column lists (keep order) ----
    feature_cols_u = list(dict.fromkeys(feature_cols))
    cols_needed = list(dict.fromkeys(feature_cols_u + [target, group_col]))

    # ---- 1) build dataframe (only filter on target/group missing) ----
    tmp = df_feat.loc[:, cols_needed].copy()
    tmp = tmp[tmp[target].notna() & tmp[group_col].notna()]

    X = tmp[feature_cols_u].copy()
    y = tmp[target].astype(int)
    g = tmp[group_col]

    # ---- 2) reproduce split (must match your modeling setup) ----
    X_tr, X_va, X_te, y_tr, y_va, y_te = split_train_val_test(
        X, y, random_state=random_state, stratify=stratify
    )
    g_te = g.loc[X_te.index]

    # ---- 3) predicted probabilities on test ----
    proba = model.predict_proba(X_te)[:, 1]

    # ---- 4) group-wise PR-AUC ----
    rows = []
    # group indices by label value
    for grp_val, idx in g_te.groupby(g_te).groups.items():
        mask = g_te.index.isin(idx)
        y_g = y_te.loc[mask].values
        p_g = proba[mask]

        # PR-AUC undefined if only one class present
        pr_auc = np.nan
        if len(np.unique(y_g)) == 2:
            pr_auc = average_precision_score(y_g, p_g)

        rows.append({
            group_col: grp_val,
            "n": int(mask.sum()),
            "prevalence": float(y_g.mean()),
            "PR_AUC": float(pr_auc) if pr_auc == pr_auc else np.nan,
        })

    out = pd.DataFrame(rows)

    # ---- 5) recall@top-k under GLOBAL top-k selection ----
    order = np.argsort(-proba)
    n = len(proba)

    for frac in top_fracs:
        k = int(round(frac * n))
        pred = np.zeros(n, dtype=int)
        pred[order[:k]] = 1

        recalls = []
        for grp_val, idx in g_te.groupby(g_te).groups.items():
            mask = g_te.index.isin(idx)
            y_g = y_te.loc[mask].values
            pred_g = pred[mask]

            if y_g.sum() == 0:
                rec = np.nan
            else:
                rec = float(pred_g[y_g == 1].sum() / y_g.sum())

            recalls.append((grp_val, rec))

        rec_col = f"recall_top{int(frac*100)}%"
        rec_df = pd.DataFrame(recalls, columns=[group_col, rec_col])
        out = out.merge(rec_df, on=group_col, how="left")

    # ---- 6) pretty formatting ----
    out["prevalence"] = out["prevalence"].round(3)
    out["PR_AUC"] = out["PR_AUC"].round(3)
    for c in out.columns:
        if c.startswith("recall_top"):
            out[c] = out[c].round(3)

    # sort: highest PR-AUC first; if NaN, push down
    out = out.sort_values(["PR_AUC", "n"], ascending=[False, False])

    return out


In [288]:
def ensure_age_group(df, age_col="AGE", out_col="AGE_GROUP"):
    if out_col in df.columns:
        return df
    df = df.copy()
    df[out_col] = pd.cut(
        df[age_col],
        bins=[0, 18, 30, 45, 65, 200],
        right=False,
        labels=["0–17", "18–29", "30–44", "45–64", "65+"]
    )
    return df

df_feat = ensure_age_group(df_feat)


In [289]:
GROUPS = ["SEX_BIN", "AGE_GROUP"]
TOP_FRACS = (0.05, 0.10, 0.20)

# HIGHCOST
for gc in GROUPS:
    display(group_fairness_audit(df_feat, "HIGHCOST_Y2", FEATURES, clf_highcost, gc, top_fracs=TOP_FRACS, stratify=True))

# ANY_ED
for gc in GROUPS:
    display(group_fairness_audit(df_feat, "ANY_ED_Y2", FEATURES, clf_ed, gc, top_fracs=TOP_FRACS, stratify=True))

# ANY_IP
for gc in GROUPS:
    display(group_fairness_audit(df_feat, "ANY_IP_Y2", FEATURES, clf_ip, gc, top_fracs=TOP_FRACS, stratify=True))


,SEX_BIN,n,prevalence,PR_AUC,recall_top5%,recall_top10%,recall_top20%
0,0,751,0.079,0.497,0.271,0.542,0.729
1,1,812,0.119,0.444,0.268,0.536,0.680


/var/folders/nr/rpw2sxkn3gqfcrlz5z0bcfw80000gp/T/ipykernel_11095/3536340463.py:50: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for grp_val, idx in g_te.groupby(g_te).groups.items():
/var/folders/nr/rpw2sxkn3gqfcrlz5z0bcfw80000gp/T/ipykernel_11095/3536340463.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for grp_val, idx in g_te.groupby(g_te).groups.items():
/var/folders/nr/rpw2sxkn3gqfcrlz5z0bcfw80000gp/T/ipykernel_11095/3536340463.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior 

,AGE_GROUP,n,prevalence,PR_AUC,recall_top5%,recall_top10%,recall_top20%
3,65+,363,0.204,0.523,0.378,0.757,0.905
2,45-64,433,0.125,0.485,0.259,0.481,0.648
0,0-17,281,0.011,0.394,0.000,0.000,0.000
1,18-44,486,0.051,0.211,0.000,0.080,0.280


,SEX_BIN,n,prevalence,PR_AUC,recall_top5%,recall_top10%,recall_top20%
1,1,841,0.161,0.365,0.222,0.363,0.496
0,0,722,0.122,0.330,0.114,0.193,0.352


/var/folders/nr/rpw2sxkn3gqfcrlz5z0bcfw80000gp/T/ipykernel_11095/3536340463.py:50: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for grp_val, idx in g_te.groupby(g_te).groups.items():
/var/folders/nr/rpw2sxkn3gqfcrlz5z0bcfw80000gp/T/ipykernel_11095/3536340463.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for grp_val, idx in g_te.groupby(g_te).groups.items():
/var/folders/nr/rpw2sxkn3gqfcrlz5z0bcfw80000gp/T/ipykernel_11095/3536340463.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior 

,AGE_GROUP,n,prevalence,PR_AUC,recall_top5%,recall_top10%,recall_top20%
2,45-64,409,0.176,0.394,0.181,0.278,0.431
3,65+,354,0.198,0.365,0.257,0.443,0.614
0,0-17,315,0.105,0.345,0.152,0.182,0.242
1,18-44,485,0.099,0.316,0.083,0.188,0.333


,SEX_BIN,n,prevalence,PR_AUC,recall_top5%,recall_top10%,recall_top20%
1,1,823,0.095,0.25,0.218,0.321,0.500
0,0,740,0.047,0.19,0.257,0.343,0.514


/var/folders/nr/rpw2sxkn3gqfcrlz5z0bcfw80000gp/T/ipykernel_11095/3536340463.py:50: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for grp_val, idx in g_te.groupby(g_te).groups.items():
/var/folders/nr/rpw2sxkn3gqfcrlz5z0bcfw80000gp/T/ipykernel_11095/3536340463.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for grp_val, idx in g_te.groupby(g_te).groups.items():
/var/folders/nr/rpw2sxkn3gqfcrlz5z0bcfw80000gp/T/ipykernel_11095/3536340463.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior 

,AGE_GROUP,n,prevalence,PR_AUC,recall_top5%,recall_top10%,recall_top20%
3,65+,338,0.151,0.286,0.373,0.510,0.745
2,45-64,410,0.076,0.277,0.226,0.323,0.516
1,18-44,528,0.053,0.091,0.000,0.036,0.107
0,0-17,287,0.010,0.018,0.000,0.000,0.000


## enviornment conda list

In [295]:
import sys, sklearn
print(sys.executable)
print(sklearn.__version__)


/opt/anaconda3/envs/meps/bin/python
1.7.2


In [296]:
import sys, sklearn, pandas, numpy, joblib, xgboost
print("python:", sys.executable)
print("sklearn:", sklearn.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("joblib:", joblib.__version__)
print("xgboost:", xgboost.__version__)


python: /opt/anaconda3/envs/meps/bin/python
sklearn: 1.7.2
pandas: 2.3.3
numpy: 2.3.5
joblib: 1.5.2
xgboost: 3.1.2
